# Notebook 03 — Candidate Universe Generation (v9)

**Responsibilities (plan §12.4):**
- preprocessing cache generation
- detector execution (classical + deep learning)
- `candidate_raw.parquet` (§3.2)
- `candidate_union.parquet` + `candidate_union_membership.parquet` (§3.3–3.4)
- `candidate_clusters.parquet` (§3.5)

**Runs only on annotated crop regions.**

## Architecture

### Mentor method (primary — UNTOUCHED)
All mentor methods are frozen exactly as settled in mentor notebook cells 33–40.

### v9 changes — empirically-tuned per-method thresholds
Every non-mentor detector was individually diagnosed on 20 crops × 5 panels
and tuned to produce counts in the target range (avg 5–15, max <50 for
well-behaved methods; accept higher for candidate-generation-oriented methods).

Key findings from v8→v9:
1. **Adaptive noise floor removed** — it compared image-space noise to
   response-space values, killing methods with different dynamic ranges.
2. **Shape gate applied selectively** — only to `rolling_ball_residual` and
   `morph_tophat` (dense-response background-subtraction methods where shape
   filtering is genuinely useful). Removed from all other methods.
3. **3 methods dropped** — `proj_local_max_norm` (norm01 saturation),
   `wavelet_msp` and `hdome` (bimodal, most panels produce 0 detections).
4. **All detectors in one execution loop** — no more "append" section.
   Spotiflow, anscombe_log, local_snr all run in the main loop.
5. **macOS dotfile filtering** + try/except on TIFF reads.

### Detectors (27 methods)
**Mentor (6, frozen):** `mentor_v1`, `mentor_v2`, `mentor_v1_k2/k4/tight`, `consensus_v2`

**Classical single-panel (17):**
`matched_filter_v2`, `local_expansion`, `restrained_psf`, `bright_rescue`,
`sk_log`, `sk_dog`, `sk_doh`, `bigfish_style`, `trackpy`,
`proj_local_max_raw`, `proj_log_norm`, `proj_log_white`,
`proj_peakbg_max`, `proj_zlocal_max`, `atrous_wavelet`, `morph_tophat`,
`multiscale_log`, `radial_symmetry`, `hessian_blobness`,
`rolling_ball_residual`, `opening_recon_residual`,
`anscombe_log`, `local_snr`

**Deep learning (1):** `spotiflow_finetuned`

**Output files are timestamped** — re-running never silently overwrites.


In [ ]:
# -------------------------------------------------------------------
# RESET — delete ALL candidate universe outputs from any previous run
# (nb03 and any other notebooks that wrote candidate tables)
# -------------------------------------------------------------------
import shutil
from pathlib import Path

REPO_ROOT_RESET = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / ".git").exists() or (p / "registries").exists())

_tables   = REPO_ROOT_RESET / "tables"
_manifest = REPO_ROOT_RESET / "artifacts" / "manifests"
_overlays = REPO_ROOT_RESET / "artifacts" / "reports" / "nb03_candidate_universe"

# All candidate-universe-related file patterns — covers nb03 outputs
# AND any residual files from earlier ad-hoc runs
_patterns = [
    "*candidate_raw*",
    "*candidate_union*",
    "*candidate_membership*",
    "*candidate_clusters*",
    "*preprocessing_cache_manifest*",
    "*detector_run_summary*",
    "*nb03*",
    "*run_manifest*",   # any run manifest in tables or manifests dir
]

deleted = []
for _dir in [_tables, _manifest]:
    if not _dir.exists():
        continue
    for _pat in _patterns:
        for _f in _dir.glob(_pat):
            if _f.is_file():
                _f.unlink()
                deleted.append(str(_f.relative_to(REPO_ROOT_RESET)))

if _overlays.exists():
    shutil.rmtree(_overlays)
    deleted.append(str(_overlays.relative_to(REPO_ROOT_RESET)) + "/ (dir)")

if deleted:
    print(f"Deleted {len(deleted)} previous candidate universe outputs:")
    for _f in sorted(deleted):
        print(" -", _f)
else:
    print("No previous outputs found — clean slate.")


In [ ]:
# -------------------------------------------------------------------
# CONFIG (v9 — empirically tuned per-method thresholds)
# -------------------------------------------------------------------
from pathlib import Path
from datetime import datetime, timezone
import os

REPO_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / ".git").exists() or (p / "registries").exists())

CFG = {
    # crop-only execution
    "execution_mode": "debug_small",
    "run_on_crops_only": True,

    # roots
    "repo_root": str(REPO_ROOT),
    "data_root_override": None,
    "data_root_candidates": [
        str(REPO_ROOT / "data"),
        str(Path.home() / "Desktop" / "bpa1"),
        str(Path.home() / "Desktop" / "BPA1"),
        str(Path.home() / "Desktop"),
    ],

    # crop registry discovery
    "crop_registry_override": None,
    "crop_registry_search_dirs": [
        str(REPO_ROOT / "annotations" / "crop_registry"),
        str(REPO_ROOT / "tables"),
        str(REPO_ROOT / "artifacts"),
        str(REPO_ROOT / "artifacts" / "manifests"),
        str(REPO_ROOT),
    ],
    "crop_registry_extensions": [".parquet", ".yaml", ".yml", ".csv"],

    # optional subset controls
    "dataset_allowlist": None,
    "well_allowlist": ["C8", "D8"],
    "crop_id_allowlist": None,
    "max_crops_total": None,

    # canonical crop panels
    "mentor_required_panels": ["C1_638", "C4_561", "C4_638", "C5_561", "C5_638"],
    "optional_context_panels": ["CODEX_638", "C1_405", "C4_405", "C5_405", "CODEX_405"],

    # image discovery
    "image_suffixes": [".tif", ".tiff"],
    "prefer_spot_detection_subtree": True,

    # ── Preprocessing ──────────────────────────────────────────────────────
    "norm_percentiles": (1, 99),
    "white_local_sigma_px": 6.0,
    "peak_bg_sigma_small_px": 1.2,
    "peak_bg_sigma_large_px": 6.0,
    "z_local_sigma_px": 6.0,
    "highpass_sigma_px": 3.0,
    "rolling_ball_radius_px": 9,
    "opening_recon_radius_px": 7,
    "wavelet_sigmas_px": [1.0, 2.0, 4.0],
    "matched_filter_sigma_px": 2.0,
    "radial_symmetry_radius_px": [1, 2, 3, 4],
    "hessian_sigma_px": 2.0,

    # ── Shape gate (v9: applied ONLY to rolling_ball_residual, morph_tophat) ─
    "shape_gate": {
        "enabled": True,
        "patch_radius": 5,
        "sigma_gradient": 1.0,
        "sigma_tensor": 2.0,
        "min_blobness": 0.15,
        "min_peak_snr": 2.0,
    },

    # ── Per-crop candidate budget (safety net) ──────────────────────────────
    "per_method_panel_budget": 80,
    "per_crop_total_budget": 5000,

    # merge / cluster
    "merge_radius_px": 5.0,
    "cluster_radius_px": 12.0,

    # ── Mentor configs (FROZEN) ────────────────────────────────────────────
    "mentor_v1": {
        "sigma": 8.0, "min_distance": 1, "threshold_space": "normalized",
        "threshold_561": 0.999, "threshold_638": 0.999,
        "log_threshold": None, "log_threshold_percentile": 70, "min_area": 1,
        "consensus_k": 3, "consensus_radius": 5,
    },
    "mentor_v2": {
        "sigma": 8.0, "min_distance": 1, "threshold_space": "normalized",
        "threshold_561": 0.999, "threshold_638": 0.999,
        "consensus_k": 3, "consensus_radius": 5,
        "exclude_border": True, "auto_norm_percentiles": (1, 99),
    },
    "mentor_v1_k2": {
        "sigma": 8.0, "min_distance": 1, "threshold_space": "normalized",
        "threshold_561": 0.999, "threshold_638": 0.999,
        "log_threshold": None, "log_threshold_percentile": 70, "min_area": 1,
        "consensus_k": 2, "consensus_radius": 5,
    },
    "mentor_v1_k4": {
        "sigma": 8.0, "min_distance": 1, "threshold_space": "normalized",
        "threshold_561": 0.999, "threshold_638": 0.999,
        "log_threshold": None, "log_threshold_percentile": 70, "min_area": 1,
        "consensus_k": 4, "consensus_radius": 5,
    },
    "mentor_v1_tight": {
        "sigma": 8.0, "min_distance": 1, "threshold_space": "normalized",
        "threshold_561": 0.999, "threshold_638": 0.999,
        "log_threshold": None, "log_threshold_percentile": 70, "min_area": 1,
        "consensus_k": 3, "consensus_radius": 3,
    },

    # ── Per-method detector configs (v9: empirically tuned) ────────────────
    # Format: { threshold params, "use_shape_gate": bool }
    # No more noise_class / adaptive noise floor.
    "detector_defaults": {
        "proj_local_max_raw":      {"min_distance": 7, "threshold_quantile": 0.9995},
        "proj_log_norm":           {"sigma": 8.0, "min_distance": 5, "threshold_quantile": 0.9993},
        "proj_log_white":          {"sigma": 8.0, "min_distance": 5, "threshold_quantile": 0.9993},
        "proj_peakbg_max":         {"min_distance": 5, "threshold_quantile": 0.9993},
        "proj_zlocal_max":         {"min_distance": 5, "threshold_abs": 6.0},
        "morph_tophat":            {"min_distance": 5, "threshold_quantile": 0.9995, "use_shape_gate": True},
        "rolling_ball_residual":   {"min_distance": 5, "threshold_quantile": 0.9995, "use_shape_gate": True},
        "opening_recon_residual":  {"min_distance": 5, "threshold_quantile": 0.9995},
        "atrous_wavelet":          {"min_distance": 5, "threshold_quantile": 0.9997},
        "radial_symmetry":         {"min_distance": 5, "threshold_quantile": 0.9997},
        "hessian_blobness":        {"min_distance": 5, "threshold_quantile": 0.9997},
        "matched_filter_v2":       {"min_distance": 5, "threshold_quantile": 0.9993},
        "local_expansion":         {"min_distance": 5, "seed_quantile": 0.9993, "expand_radius": 4},
        "restrained_psf":          {"seed_sigma": 8.0, "min_distance": 5,
                                    "threshold_quantile": 0.9995, "fit_radius": 6},
        "bright_rescue":           {"min_distance": 5, "threshold_quantile": 0.9997},
        "bigfish_style":           {"sigma": 8.0, "min_distance": 5,
                                    "threshold_quantile": 0.9993, "cap": 30},
        "trackpy":                 {"min_distance": 5, "threshold_quantile": 0.9993,
                                    "bandpass_lsig": 1.5, "bandpass_hsig": 8.0},
        "sk_log":                  {"min_sigma": 3.0, "max_sigma": 8.0, "num_sigma": 4, "threshold": 0.20},
        "sk_dog":                  {"min_sigma": 3.0, "max_sigma": 8.0, "threshold": 0.15},
        "sk_doh":                  {"min_sigma": 3.0, "max_sigma": 8.0, "threshold": 0.02},
        "multiscale_log":          {"sigmas": [3.0, 5.0, 8.0, 10.0], "min_distance": 5,
                                    "threshold_quantile": 0.9993},
        "consensus_v2":            {"panel_consensus_k": 2, "panel_consensus_radius": 2,
                                    "global_consensus_k": 3, "global_consensus_radius": 5,
                                    "primitive_threshold_quantile": 0.9993},
        "anscombe_log":            {"sigma": 8.0, "min_distance": 5, "threshold_quantile": 0.999},
        "local_snr":               {"min_distance": 5, "snr_min": 8.0},
    },

    # outputs
    "tables_subdir": "tables",
    "artifacts_subdir": "artifacts/reports/nb03_candidate_universe",
    "manifest_subdir": "artifacts/manifests",
    "write_candidate_raw": True,
    "write_candidate_union": True,
    "write_candidate_membership": True,
    "write_candidate_clusters": True,
    "write_cache_manifest": True,
    "write_detector_summary": True,
    "write_overlays": True,

    # provenance
    "project_config_version": "nb03_v9",
    "preprocessing_registry_version": "v9_nb03",
    "detector_registry_version": "v9_nb03_empirical",
    "feature_registry_version": "v6_plan_frozen_reference_only",
    "annotation_registry_version": "v6_nb02",
    "matching_registry_version": "v6_frozen",
}

print("REPO_ROOT:", REPO_ROOT)
print("execution_mode:", CFG["execution_mode"])


In [ ]:
# -------------------------------------------------------------------
# IMPORTS
# -------------------------------------------------------------------
import gc
import json
import math
import os
import re
import time
import hashlib
from dataclasses import dataclass
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict, Counter
from typing import Optional

import numpy as np
import pandas as pd
import tifffile
import yaml
import matplotlib.pyplot as plt

from scipy import ndimage as ndi
from scipy.optimize import curve_fit
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

from skimage import feature, exposure, restoration, measure, morphology, segmentation
from skimage.feature import peak_local_max
from skimage.morphology import disk, binary_dilation, reconstruction
from skimage.filters import gaussian, laplace, threshold_otsu
from skimage.restoration import rolling_ball

print("imports ok")

In [ ]:
# -------------------------------------------------------------------
# LOGGING / PROVENANCE / IO HELPERS
# -------------------------------------------------------------------
def ts_utc() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

def log(msg: str) -> None:
    print(f"[{ts_utc()}] {msg}", flush=True)

def ensure_dir(path: str | Path) -> Path:
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return p

def make_run_id(prefix: str = "nb03") -> str:
    t = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    h = hashlib.sha1(f"{t}|{os.getpid()}".encode("utf-8")).hexdigest()[:8]
    return f"{prefix}_{t}_{h}"

RUN_ID = make_run_id("nb03")
TABLES_DIR = ensure_dir(REPO_ROOT / CFG["tables_subdir"])
ARTIFACT_DIR = ensure_dir(REPO_ROOT / CFG["artifacts_subdir"])
MANIFEST_DIR = ensure_dir(REPO_ROOT / CFG["manifest_subdir"])
OVERLAY_DIR = ensure_dir(ARTIFACT_DIR / "overlays")

def safe_to_parquet(df: pd.DataFrame, path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        df.to_parquet(path, index=False)
        return path
    except Exception as e:
        csv_path = path.with_suffix(".csv")
        log(f"[warn] parquet write failed for {path.name}: {type(e).__name__}: {e}. Writing CSV fallback -> {csv_path.name}")
        df.to_csv(csv_path, index=False)
        return csv_path

def write_json(obj, path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2), encoding="utf-8")
    return path

def sha1_text(x: str) -> str:
    return hashlib.sha1(x.encode("utf-8")).hexdigest()

def sha1_file(path: str | Path, chunk_size: int = 1 << 20) -> str:
    h = hashlib.sha1()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def get_git_commit(repo_root: Path) -> Optional[str]:
    git_dir = repo_root / ".git"
    if not git_dir.exists():
        return None
    head = git_dir / "HEAD"
    if not head.exists():
        return None
    content = head.read_text(encoding="utf-8").strip()
    if content.startswith("ref:"):
        ref = content.split(" ", 1)[1].strip()
        ref_path = git_dir / ref
        if ref_path.exists():
            return ref_path.read_text(encoding="utf-8").strip()
        return None
    return content

CODE_GIT_COMMIT = get_git_commit(REPO_ROOT)

BASE_PROVENANCE = {
    "run_id": RUN_ID,
    "project_config_version": CFG["project_config_version"],
    "preprocessing_registry_version": CFG["preprocessing_registry_version"],
    "detector_registry_version": CFG["detector_registry_version"],
    "feature_registry_version": CFG["feature_registry_version"],
    "annotation_registry_version": CFG["annotation_registry_version"],
    "matching_registry_version": CFG["matching_registry_version"],
    "code_git_commit": CODE_GIT_COMMIT,
    "merge_radius_px": float(CFG["merge_radius_px"]),
    "truth_match_radius_px": None,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

log(f"RUN_ID = {RUN_ID}")
log(f"CODE_GIT_COMMIT = {CODE_GIT_COMMIT}")

In [ ]:
# -------------------------------------------------------------------
# DISCOVERY: crop registry + image inventory
# -------------------------------------------------------------------
WELL_PAT = re.compile(r"(?<![A-Z0-9])([A-H](?:[1-9]|1[0-2]))(?![A-Z0-9])", re.IGNORECASE)
CHANNEL_PAT = re.compile(r"(?:Fluorescence_)?(405|561|638)_nm_Ex", re.IGNORECASE)
CYCLE_PAT = re.compile(r"(Cycle\d+|CODEX)", re.IGNORECASE)

def choose_data_root(candidates, override=None) -> Path:
    if override:
        p = Path(override).expanduser().resolve()
        if not p.exists():
            raise FileNotFoundError(f"Configured data_root_override does not exist: {p}")
        return p
    existing = []
    for cand in candidates:
        p = Path(cand).expanduser().resolve()
        if p.exists():
            n_tif = sum(1 for x in p.rglob("*") if x.is_file() and x.suffix.lower() in {".tif", ".tiff"})
            existing.append((n_tif, p))
    if not existing:
        raise FileNotFoundError("No existing data-root candidates found.")
    existing.sort(key=lambda t: (t[0], -len(str(t[1]))), reverse=True)
    return existing[0][1]

def choose_latest_file(search_dirs, suffixes, override=None) -> Path:
    """
    Find the most recently modified file matching any of the given suffixes
    across the search directories. Prefers crop_registry files specifically.
    Handles NB01's YAML output (e.g. nb01_..._crop_registry.yaml) and
    parquet outputs.
    """
    if override:
        p = Path(override).expanduser().resolve()
        if not p.exists():
            raise FileNotFoundError(f"Configured crop_registry_override does not exist: {p}")
        return p
    found = []
    for d in search_dirs:
        dd = Path(d).expanduser().resolve()
        if not dd.exists():
            continue
        for suf in suffixes:
            found.extend(dd.rglob(f"*{suf}"))
    found = [p for p in found if p.is_file()]
    if not found:
        # Build a helpful error message showing what was searched
        searched = [str(Path(d).expanduser().resolve()) for d in search_dirs]
        raise FileNotFoundError(
            f"No crop registry file found.\n"
            f"Searched directories: {searched}\n"
            f"Searched suffixes: {suffixes}\n"
            f"Run Notebook 01 first to generate the crop registry."
        )
    # Prefer files with "crop_registry" in name; within that group prefer most recent
    registry_files = [p for p in found if "crop_registry" in p.name.lower()]
    candidates = registry_files if registry_files else found
    candidates.sort(key=lambda p: (p.stat().st_mtime, p.name))
    return candidates[-1]

def load_crop_registry(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    suf = path.suffix.lower()
    if suf == ".parquet":
        return pd.read_parquet(path)
    if suf in {".yaml", ".yml"}:
        payload = yaml.safe_load(path.read_text(encoding="utf-8"))
        if isinstance(payload, dict) and "records" in payload:
            return pd.DataFrame(payload["records"])
        if isinstance(payload, list):
            return pd.DataFrame(payload)
        raise ValueError(f"Unsupported YAML crop registry structure in {path}")
    if suf == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported crop registry suffix: {suf}")

def normalize_crop_registry(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "selection_tags" in out.columns:
        out["selection_tags"] = out["selection_tags"].apply(
            lambda x: list(x) if isinstance(x, (list, tuple, set)) else ([] if pd.isna(x) else [str(x)])
        )
    for c in ["well_ymin_px","well_xmin_px","well_ymax_px","well_xmax_px"]:
        out[c] = out[c].astype(int)
    return out

def inventory_images(root: Path, suffixes=(".tif", ".tiff")) -> pd.DataFrame:
    rows = []
    for p in root.rglob("*"):
        if not p.is_file():
            continue
        if p.suffix.lower() not in {s.lower() for s in suffixes}:
            continue
        # Skip macOS resource-fork dotfiles (._*) which are not valid TIFFs
        if p.name.startswith("._"):
            continue
        path_str = str(p)
        well_match = WELL_PAT.search(" ".join(p.parts[-5:]) + " " + p.name)
        cycle_match = CYCLE_PAT.search(" ".join(p.parts[-5:]) + " " + p.name)
        ch_match = CHANNEL_PAT.search(p.name)
        well_id = well_match.group(1).upper() if well_match else None
        cycle_key = cycle_match.group(1).upper() if cycle_match else None
        channel_nm = ch_match.group(1) if ch_match else None
        rows.append({
            "file_path": str(p.resolve()),
            "file_name": p.name,
            "well_id": well_id,
            "cycle_key": cycle_key,
            "channel_nm": channel_nm,
            "parent": p.parent.name,
            "path_depth": len(p.parts),
        })
    df = pd.DataFrame(rows)
    if len(df) == 0:
        return pd.DataFrame(columns=["file_path","file_name","well_id","cycle_key","channel_nm","parent","path_depth"])
    return df.sort_values(["well_id","cycle_key","channel_nm","file_name"], na_position="last").reset_index(drop=True)

def panel_key_from_cycle_channel(cycle_key, channel_nm):
    if cycle_key is None or channel_nm is None:
        return None
    cycle_key = cycle_key.upper()
    if cycle_key.startswith("CYCLE"):
        num = cycle_key.replace("CYCLE", "")
        return f"C{num}_{channel_nm}"
    if cycle_key == "CODEX":
        return f"CODEX_{channel_nm}"
    return None

DATA_ROOT = choose_data_root(CFG["data_root_candidates"], CFG["data_root_override"])
CROP_REGISTRY_PATH = choose_latest_file(
    CFG["crop_registry_search_dirs"],
    CFG["crop_registry_extensions"],
    CFG["crop_registry_override"],
)

log(f"Selected DATA_ROOT = {DATA_ROOT}")
log(f"Selected CROP_REGISTRY_PATH = {CROP_REGISTRY_PATH}")

crop_registry = normalize_crop_registry(load_crop_registry(CROP_REGISTRY_PATH))
if CFG["dataset_allowlist"] is not None:
    crop_registry = crop_registry[crop_registry["dataset_id"].isin(CFG["dataset_allowlist"])].copy()
if CFG["well_allowlist"] is not None:
    crop_registry = crop_registry[crop_registry["well_id"].isin(CFG["well_allowlist"])].copy()
if CFG["crop_id_allowlist"] is not None:
    crop_registry = crop_registry[crop_registry["crop_id"].isin(CFG["crop_id_allowlist"])].copy()
if CFG["max_crops_total"] is not None:
    crop_registry = crop_registry.head(int(CFG["max_crops_total"])).copy()

if len(crop_registry) == 0:
    raise RuntimeError("Crop registry is empty after filtering.")

image_inventory = inventory_images(DATA_ROOT, CFG["image_suffixes"])
if CFG["well_allowlist"] is not None:
    image_inventory = image_inventory[image_inventory["well_id"].isin(CFG["well_allowlist"])].copy()
image_inventory["panel_key"] = [
    panel_key_from_cycle_channel(cyc, ch)
    for cyc, ch in zip(image_inventory["cycle_key"], image_inventory["channel_nm"])
]

display(crop_registry.head())
display(image_inventory.head())
log(f"Crop count = {len(crop_registry)}")
log(f"Image inventory rows = {len(image_inventory)}")

In [ ]:
# -------------------------------------------------------------------
# PANEL RESOLUTION FOR EACH CROP
# -------------------------------------------------------------------
def choose_one_file(group: pd.DataFrame) -> str:
    g = group.sort_values(
        by=["path_depth", "file_name", "file_path"],
        ascending=[True, True, True],
        na_position="last",
    )
    return str(g.iloc[0]["file_path"])

panel_file_map = {}
missing_panel_report = []

for _, crop in crop_registry.iterrows():
    crop_id = crop["crop_id"]
    well_id = crop["well_id"]
    inv = image_inventory[image_inventory["well_id"] == well_id].copy()
    inv = inv[inv["panel_key"].notna()].copy()

    chosen = {}
    for panel_key, grp in inv.groupby("panel_key", dropna=False):
        if panel_key is None:
            continue
        chosen[str(panel_key)] = choose_one_file(grp)

    panel_file_map[crop_id] = chosen

    missing = [p for p in CFG["mentor_required_panels"] if p not in chosen]
    missing_panel_report.append({
        "crop_id": crop_id,
        "well_id": well_id,
        "missing_required_panels": missing,
        "n_available_panels": len(chosen),
    })

missing_panel_df = pd.DataFrame(missing_panel_report)
display(missing_panel_df)

n_missing = int(missing_panel_df["missing_required_panels"].apply(len).gt(0).sum())
if n_missing:
    log(f"[warn] {n_missing} crop(s) do not have all mentor-required panels. mentor methods will be skipped for those crops.")
else:
    log("All crops have all mentor-required panels.")

In [ ]:
# -------------------------------------------------------------------
# CORE IMAGE / GEOMETRY HELPERS
# -------------------------------------------------------------------
def read_image_2d(path: str | Path) -> np.ndarray:
    """
    Read a 2D float32 image from path.
    Raises TiffFileError/ValueError for unreadable files so callers
    can skip them with a warning (handles macOS ._* dotfiles etc.).
    """
    arr = tifffile.imread(str(path))
    arr = np.asarray(arr)
    arr = np.squeeze(arr)
    if arr.ndim != 2:
        # Multi-channel or multi-Z: max-project to 2D rather than crashing
        if arr.ndim == 3:
            arr = arr.max(axis=0)
        else:
            arr = arr.reshape(arr.shape[-2], arr.shape[-1])
    return arr.astype(np.float32, copy=False)

def crop_image_by_registry(img: np.ndarray, crop_row: pd.Series) -> np.ndarray:
    y0 = int(crop_row["well_ymin_px"])
    y1 = int(crop_row["well_ymax_px"])
    x0 = int(crop_row["well_xmin_px"])
    x1 = int(crop_row["well_xmax_px"])
    return img[y0:y1, x0:x1]

def norm01(img, p_lo=1, p_hi=99):
    lo = float(np.percentile(img, p_lo))
    hi = float(np.percentile(img, p_hi))
    if hi <= lo:
        hi = lo + 1.0
    x = (img - lo) / (hi - lo)
    return np.clip(x, 0, 1).astype(np.float32), lo, hi

def norm01_with_limits(img, vmin=None, vmax=None, p_lo=1, p_hi=99):
    if vmin is None or vmax is None:
        _, vmin, vmax = norm01(img, p_lo, p_hi)
    x = (img - vmin) / max(vmax - vmin, 1e-6)
    return np.clip(x, 0, 1).astype(np.float32), float(vmin), float(vmax)

def zscore_local(img, sigma):
    mu = ndi.gaussian_filter(img, sigma=sigma)
    mu2 = ndi.gaussian_filter(img * img, sigma=sigma)
    var = np.maximum(mu2 - mu * mu, 1e-6)
    return ((img - mu) / np.sqrt(var)).astype(np.float32)

def white_local(img, sigma):
    mu = ndi.gaussian_filter(img, sigma=sigma)
    resid = img - mu
    var = ndi.gaussian_filter(resid * resid, sigma=sigma)
    return (resid / np.sqrt(np.maximum(var, 1e-6))).astype(np.float32)

def peak_bg(img, sigma_small, sigma_large):
    small = ndi.gaussian_filter(img, sigma=sigma_small)
    large = ndi.gaussian_filter(img, sigma=sigma_large)
    return (small - large).astype(np.float32)

def highpass(img, sigma):
    return (img - ndi.gaussian_filter(img, sigma=sigma)).astype(np.float32)

def local_mad_map(img, radius=5):
    med = ndi.median_filter(img, size=2 * radius + 1)
    mad = ndi.median_filter(np.abs(img - med), size=2 * radius + 1)
    return (mad / 0.6745 + 1e-6).astype(np.float32)

def white_transform(img, sigma):
    hp = highpass(img, sigma)
    mad = local_mad_map(hp, radius=max(2, int(round(sigma))))
    out = hp / mad
    return np.clip(out, -8, 8).astype(np.float32)

def opening_by_reconstruction_residual(img, radius):
    se = morphology.disk(int(radius))
    eroded = morphology.erosion(img, se)
    opened = reconstruction(eroded, img, method="dilation")
    return (img - opened).astype(np.float32)

def rolling_ball_residual(img, radius):
    bg = rolling_ball(img, radius=radius)
    return (img - bg).astype(np.float32)

def atrous_wavelet_bandpass(img, sigmas):
    acc = np.zeros_like(img, dtype=np.float32)
    prev = img.astype(np.float32)
    for s in sigmas:
        blur = ndi.gaussian_filter(img, sigma=s).astype(np.float32)
        detail = np.maximum(prev - blur, 0)
        acc += detail
        prev = blur
    return acc.astype(np.float32)

def wavelet_product_map(img, sigmas):
    details = []
    prev = img.astype(np.float32)
    for s in sigmas:
        blur = ndi.gaussian_filter(img, sigma=s).astype(np.float32)
        details.append(np.maximum(prev - blur, 0))
        prev = blur
    prod = np.ones_like(img, dtype=np.float32)
    for d in details:
        prod *= (d + 1e-6)
    prod = np.power(prod, 1.0 / max(len(details), 1))
    return prod.astype(np.float32)

def hessian_blobness_map(img, sigma):
    sm = ndi.gaussian_filter(img, sigma=sigma).astype(np.float32)
    Iyy, Iyx = np.gradient(np.gradient(sm, axis=0), axis=0), np.gradient(np.gradient(sm, axis=0), axis=1)
    Ixy, Ixx = np.gradient(np.gradient(sm, axis=1), axis=0), np.gradient(np.gradient(sm, axis=1), axis=1)
    tr = Ixx + Iyy
    det = Ixx * Iyy - Ixy * Iyx
    disc = np.maximum(tr * tr - 4.0 * det, 0.0)
    lam1 = 0.5 * (tr + np.sqrt(disc))
    lam2 = 0.5 * (tr - np.sqrt(disc))
    blob = np.abs(lam1 * lam2)
    return blob.astype(np.float32)

def radial_symmetry_map(img, radii=(1, 2, 3, 4)):
    """
    Gradient-vote radial symmetry map at multiple radii.
    Voting at multiple radii (RS-FISH: Bahry et al. 2022, Nature Methods)
    gives a more robust response than a single radius.
    """
    img = img.astype(np.float32)
    gy, gx = np.gradient(ndi.gaussian_filter(img, sigma=1.0))
    mag = np.sqrt(gx * gx + gy * gy) + 1e-6
    gx_n = gx / mag
    gy_n = gy / mag
    H, W = img.shape
    ys, xs = np.indices((H, W))
    accum = np.zeros((H, W), dtype=np.float32)
    for radius in radii:
        pos_y = np.clip(np.round(ys + radius * gy_n).astype(int), 0, H - 1)
        pos_x = np.clip(np.round(xs + radius * gx_n).astype(int), 0, W - 1)
        np.add.at(accum, (pos_y.ravel(), pos_x.ravel()), mag.ravel())
    return ndi.gaussian_filter(accum, sigma=1.0).astype(np.float32)

def normalized_log(img, sigma):
    return (-(sigma ** 2) * ndi.gaussian_laplace(img.astype(np.float32), sigma=sigma)).astype(np.float32)

def matched_filter_response(img, sigma):
    kernel = feature.match_template(ndi.gaussian_filter(np.zeros((21, 21), dtype=np.float32), sigma=0), np.zeros((3,3)))  # no-op placeholder to keep skimage loaded
    # explicit gaussian kernel
    rad = int(max(3, round(4 * sigma)))
    yy, xx = np.mgrid[-rad:rad+1, -rad:rad+1]
    g = np.exp(-(xx*xx + yy*yy) / (2.0 * sigma * sigma)).astype(np.float32)
    g /= np.sqrt(np.sum(g * g)) + 1e-6
    img0 = img.astype(np.float32) - float(np.mean(img))
    resp = ndi.correlate(img0, g, mode="nearest")
    return resp.astype(np.float32)

def border_distance_map(shape):
    H, W = shape
    ys, xs = np.indices((H, W))
    return np.minimum.reduce([ys, xs, H - 1 - ys, W - 1 - xs]).astype(np.float32)

def local_max_points(resp, min_distance=3, threshold_abs=None, threshold_rel=None):
    coords = peak_local_max(
        resp,
        min_distance=int(min_distance),
        threshold_abs=threshold_abs,
        threshold_rel=threshold_rel,
        exclude_border=False,
    )
    if coords.size == 0:
        return np.zeros((0, 2), dtype=float)
    return coords.astype(float)

def robust_threshold_from_quantile(resp, q):
    finite = np.asarray(resp[np.isfinite(resp)], dtype=np.float32)
    if finite.size == 0:
        return np.inf
    return float(np.quantile(finite, q))

def minmax01(vals):
    vals = np.asarray(vals, dtype=np.float32)
    if vals.size == 0:
        return vals
    lo = float(np.min(vals))
    hi = float(np.max(vals))
    if hi <= lo:
        return np.ones_like(vals, dtype=np.float32)
    return ((vals - lo) / (hi - lo)).astype(np.float32)

def sample_scores(resp, coords):
    if len(coords) == 0:
        return np.array([], dtype=np.float32)
    yy = np.clip(np.round(coords[:, 0]).astype(int), 0, resp.shape[0] - 1)
    xx = np.clip(np.round(coords[:, 1]).astype(int), 0, resp.shape[1] - 1)
    return resp[yy, xx].astype(np.float32)

def make_detection_df(
    coords_crop_yx,
    scores_raw,
    *,
    run_id,
    dataset_id,
    well_id,
    crop_id,
    method_id,
    method_family,
    source_view_id,
    score_type,
    score_is_calibrated,
    sigma_px,
    timing_sec,
    crop_origin_well_y_px,
    crop_origin_well_x_px,
    image_shape_y,
    image_shape_x,
    coord_semantics,
    source_image_fingerprint,
):
    coords = np.asarray(coords_crop_yx, dtype=np.float32).reshape(-1, 2)
    scores_raw = np.asarray(scores_raw, dtype=np.float32).reshape(-1)
    if len(coords) != len(scores_raw):
        raise ValueError(f"coords/scores length mismatch: {len(coords)} vs {len(scores_raw)}")
    scores_norm = minmax01(scores_raw)
    rows = []
    for i, ((cy, cx), sraw, snorm) in enumerate(zip(coords, scores_raw, scores_norm)):
        wy = float(crop_origin_well_y_px + cy)
        wx = float(crop_origin_well_x_px + cx)
        raw_id = sha1_text(f"{run_id}|{dataset_id}|{well_id}|{crop_id}|{method_id}|{source_view_id}|{i:08d}")
        rows.append({
            "raw_detection_id": raw_id,
            "run_id": run_id,
            "dataset_id": dataset_id,
            "well_id": well_id,
            "crop_id": crop_id,
            "method_id": method_id,
            "method_family": method_family,
            "source_view_id": source_view_id,
            "score_type": score_type,
            "score_is_calibrated": bool(score_is_calibrated),
            "detection_score_raw": float(sraw),
            "detection_score_norm": float(snorm),
            "crop_y_px": float(cy),
            "crop_x_px": float(cx),
            "well_y_px": wy,
            "well_x_px": wx,
            "coord_frame_primary": "well",
            "coord_units": "pixel",
            "sigma_px": None if sigma_px is None else float(sigma_px),
            "timing_sec": float(timing_sec),
            "crop_origin_well_y_px": int(crop_origin_well_y_px),
            "crop_origin_well_x_px": int(crop_origin_well_x_px),
            "image_shape_y": int(image_shape_y),
            "image_shape_x": int(image_shape_x),
            "coordinate_semantics": coord_semantics,
            "source_image_fingerprint": source_image_fingerprint,
            **BASE_PROVENANCE,
        })
    return pd.DataFrame(rows)

log("Core helpers defined.")

# ===================================================================
# v8 ADAPTIVE THRESHOLD INFRASTRUCTURE
# ===================================================================

def estimate_panel_noise_sigma(cache: dict) -> float:
    """
    Estimate the per-panel noise standard deviation from the local MAD map.

    The local MAD map (already in cache) measures pixel-level noise robustly.
    We take the median of the MAD map as the representative noise sigma for
    this panel.  This is insensitive to sparse bright spots (unlike std) and
    captures spatial heterogeneity better than a single global estimate.

    Returns sigma_noise in [0, 1] normalised image units.
    """
    mad_map = cache.get("local_mad", None)
    if mad_map is None:
        # Fallback: estimate from norm01 image directly
        img = cache["norm01"]
        med = ndi.median_filter(img, size=11)
        residual = img - med
        return float(np.median(np.abs(residual)) / 0.6745 + 1e-8)
    # local_mad is already scaled by 1/0.6745 in the preprocessing
    return float(np.median(mad_map))


def adaptive_threshold(resp: np.ndarray, cache: dict, *,
                       threshold_quantile: float = 0.9993,
                       noise_class: str = "dense") -> float:
    """
    Two-stage adaptive threshold for non-mentor detectors.

    Stage 1: quantile-based seed threshold (as before).
    Stage 2: noise-aware floor derived from the panel noise estimate.
    Final = max(stage1, stage2).

    noise_class controls the noise multiplier:
      "dense"  → CFG["adaptive_threshold"]["noise_k_dense"]  (strict)
      "sparse" → CFG["adaptive_threshold"]["noise_k_sparse"] (moderate)
      "rescue" → CFG["adaptive_threshold"]["noise_k_rescue"] (very strict)
      "calibrated" → skip adaptive (detector has its own calibration)
    """
    if noise_class == "calibrated":
        # Detector already has a physically-calibrated threshold (e.g. z-score)
        return robust_threshold_from_quantile(resp, threshold_quantile)

    seed_thr = robust_threshold_from_quantile(resp, threshold_quantile)

    sigma_noise = estimate_panel_noise_sigma(cache)
    k_map = CFG["adaptive_threshold"]
    if noise_class == "dense":
        k = k_map["noise_k_dense"]
    elif noise_class == "rescue":
        k = k_map["noise_k_rescue"]
    else:
        k = k_map["noise_k_sparse"]

    noise_floor = k * sigma_noise
    return max(seed_thr, noise_floor)


# ===================================================================
# v8 MORPHOLOGICAL SHAPE GATE
# ===================================================================

def shape_gate_filter(img: np.ndarray, coords: np.ndarray,
                      scores: np.ndarray, cache: dict = None) -> tuple:
    """
    Post-detection filter: rejects points that lack blob-like curvature.

    Uses the structure tensor (outer product of image gradients, smoothed)
    in a local patch around each detection.  The eigenvalue ratio
    lambda_min / lambda_max measures local isotropy:
      ratio ≈ 1  → circular blob (keep)
      ratio ≈ 0  → edge or fibre (reject)
      ratio = NaN → flat region / noise (reject)

    Also rejects detections where the peak value is < min_peak_snr
    times the local background (estimated as the patch median).

    This filter is NOT applied to mentor methods.

    Returns (filtered_coords, filtered_scores).
    """
    cfg_sg = CFG["shape_gate"]
    if not cfg_sg["enabled"] or len(coords) == 0:
        return coords, scores

    patch_r    = cfg_sg["patch_radius"]
    sigma_g    = cfg_sg["sigma_gradient"]
    sigma_t    = cfg_sg["sigma_tensor"]
    min_blob   = cfg_sg["min_blobness"]
    min_snr    = cfg_sg["min_peak_snr"]

    H, W = img.shape
    keep_mask = np.ones(len(coords), dtype=bool)

    # Pre-compute image gradients (once for all detections)
    img_smooth = ndi.gaussian_filter(img.astype(np.float64), sigma=sigma_g)
    gy, gx = np.gradient(img_smooth)

    # Structure tensor components (smoothed outer products)
    Jxx = ndi.gaussian_filter(gx * gx, sigma=sigma_t)
    Jyy = ndi.gaussian_filter(gy * gy, sigma=sigma_t)
    Jxy = ndi.gaussian_filter(gx * gy, sigma=sigma_t)

    for i, (cy, cx) in enumerate(coords):
        iy, ix = int(round(cy)), int(round(cx))

        # Bounds check
        y0 = max(0, iy - patch_r)
        y1 = min(H, iy + patch_r + 1)
        x0 = max(0, ix - patch_r)
        x1 = min(W, ix + patch_r + 1)

        if (y1 - y0) < 3 or (x1 - x0) < 3:
            keep_mask[i] = False
            continue

        # Local structure tensor (average over patch)
        jxx = float(np.mean(Jxx[y0:y1, x0:x1]))
        jyy = float(np.mean(Jyy[y0:y1, x0:x1]))
        jxy = float(np.mean(Jxy[y0:y1, x0:x1]))

        # Eigenvalues of 2×2 structure tensor
        trace = jxx + jyy
        det_val = jxx * jyy - jxy * jxy
        disc = max(trace * trace - 4.0 * det_val, 0.0)
        lam1 = 0.5 * (trace + math.sqrt(disc))
        lam2 = 0.5 * (trace - math.sqrt(disc))
        lam_max = max(abs(lam1), abs(lam2))
        lam_min = min(abs(lam1), abs(lam2))

        # Blobness = lambda_min / lambda_max  ∈ [0, 1]
        if lam_max < 1e-12:
            # Flat region — no structure at all
            keep_mask[i] = False
            continue

        blobness = lam_min / lam_max
        if blobness < min_blob:
            keep_mask[i] = False
            continue

        # Local SNR check: peak vs. local background
        patch = img[y0:y1, x0:x1]
        local_bg = float(np.median(patch))
        peak_val = float(img[min(iy, H-1), min(ix, W-1)])
        if local_bg > 0 and (peak_val / local_bg) < min_snr:
            keep_mask[i] = False
            continue

    n_rejected = int((~keep_mask).sum())
    if n_rejected > 0 and len(coords) > 0:
        pass  # Logged by caller if desired

    return coords[keep_mask], scores[keep_mask]


def apply_per_method_budget(detections: list, crop_id: str) -> list:
    """
    Enforce per-method-panel budget: keep at most N detections per
    (method_id, source_view_id) combination, ranked by detection_score_raw.

    Also enforce per-crop total budget.

    Returns a pruned list of DataFrames.
    """
    budget_mp = CFG["per_method_panel_budget"]
    budget_total = CFG["per_crop_total_budget"]

    pruned = []
    total_before = 0
    total_after = 0

    for df in detections:
        if len(df) == 0:
            pruned.append(df)
            continue
        total_before += len(df)

        # Per method-panel budget
        if len(df) > budget_mp:
            df = df.nlargest(budget_mp, "detection_score_raw")

        pruned.append(df)
        total_after += len(df)

    # Total budget enforcement: if still over, rank all and keep top-N
    if total_after > budget_total and len(pruned) > 0:
        combined = pd.concat(pruned, ignore_index=True)
        combined = combined.nlargest(budget_total, "detection_score_raw")
        # Rebuild per-method DFs for consistency
        pruned = [combined]
        total_after = len(combined)

    if total_before != total_after:
        log(f"  [budget] crop={crop_id[-12:]} pruned {total_before} → {total_after} raw detections")

    return pruned


# ===================================================================
# v8 PROPERLY-THRESHOLDED WAVELET MULTISCALE PRODUCT (main loop)
# ===================================================================
# Replaces the naive wavelet_product which did NOT threshold per-scale.
# This is the correct WMP per Olivo-Marin (2002).

def wavelet_msp_response(norm_img: np.ndarray,
                         n_scales: int = 3,
                         k_sigma: float = 5.0) -> np.ndarray:
    """
    Wavelet Multiscale Product with per-scale noise thresholding.

    At each scale j, detail coefficients w_j are hard-thresholded at
    k * sigma_j, where sigma_j = MAD(w_j) / 0.6745 (robust noise estimate).
    Thresholded details are multiplied across scales and geometric-mean
    normalised.

    Ref: Olivo-Marin (2002) Pattern Recognition 35(9):1989-1996.
    """
    img = norm_img.astype(np.float64)
    approx = img.copy()
    _b3_coeffs = [0.8909, 0.2007, 0.0856, 0.0428, 0.0215]
    product = None

    for j in range(1, n_scales + 1):
        sigma_j = 2.0 ** (j - 1)
        next_approx = ndi.gaussian_filter(approx, sigma=sigma_j)
        w_j = approx - next_approx

        b3 = _b3_coeffs[min(j - 1, len(_b3_coeffs) - 1)]
        mad_j = float(np.median(np.abs(w_j))) / 0.6745
        sigma_noise_j = max(mad_j / b3, 1e-10)
        thr_j = k_sigma * sigma_noise_j
        w_j_thresh = np.where(np.abs(w_j) >= thr_j, np.abs(w_j), 0.0)

        if product is None:
            product = w_j_thresh.copy()
        else:
            product *= (w_j_thresh + 1e-30)
        approx = next_approx

    wmp_img = np.power(np.clip(product, 0, None), 1.0 / n_scales).astype(np.float32)
    return wmp_img


In [ ]:
# -------------------------------------------------------------------
# MENTOR METHOD REPRODUCTION HELPERS
# -------------------------------------------------------------------
def log_spot_mask_mentor_v1(
    img,
    sigma=3.0,
    min_distance=12,
    threshold=0.5,
    threshold_space="normalized",
    p_lo=1,
    p_hi=99,
    log_threshold=None,
    log_threshold_percentile=90,
    min_area=0,
):
    img = img.astype(np.float32, copy=False)
    if threshold_space == "normalized":
        work, _, _ = norm01(img, p_lo=p_lo, p_hi=p_hi)
        thr_int = float(threshold)
    else:
        work = img
        thr_int = float(threshold)
    clog = -ndi.gaussian_laplace(work, sigma=sigma)
    size = 2 * int(min_distance) + 1
    maxf = ndi.maximum_filter(clog, size=size, mode="nearest")
    candidate_peaks = (clog == maxf) & (work > thr_int)
    if not np.any(candidate_peaks):
        return np.zeros_like(img, dtype=bool), clog.astype(np.float32)
    if log_threshold is None:
        peak_vals = clog[candidate_peaks]
        log_thr = np.percentile(peak_vals, log_threshold_percentile)
    else:
        log_thr = float(log_threshold)
    mask = candidate_peaks & (clog >= log_thr)
    if min_area > 0:
        lbl = measure.label(mask)
        keep_mask = np.zeros_like(mask, dtype=bool)
        for r in measure.regionprops(lbl):
            if r.area >= min_area:
                keep_mask[lbl == r.label] = True
        mask = keep_mask
    return mask, clog.astype(np.float32)

def consensus_from_masks(masks, tolerance=2, min_hits=4):
    se = disk(int(tolerance)) if tolerance and tolerance > 0 else None
    dilated = [binary_dilation(m, se) if se is not None else m for m in masks]
    count = np.zeros_like(masks[0], dtype=np.uint16)
    for m in dilated:
        count += m.astype(np.uint16)
    cons_mask = count >= int(min_hits)
    lbl = measure.label(cons_mask)
    pts = []
    for r in measure.regionprops(lbl):
        cy, cx = r.centroid
        pts.append((float(cy), float(cx)))
    return cons_mask, np.asarray(pts, dtype=np.float32).reshape(-1, 2), count.astype(np.float32)

def quick_percentile_limits_np(img, p_lo=1, p_hi=99, step=16):
    samp = img[::step, ::step]
    vmin = float(np.percentile(samp, p_lo))
    vmax = float(np.percentile(samp, p_hi))
    if vmax <= vmin:
        vmax = vmin + 1.0
    return vmin, vmax

def log_peaks_2d_mentor_v2(
    img2d,
    sigma=3.0,
    min_distance=12,
    threshold=None,
    exclude_border=True,
    *,
    threshold_space="raw",
    norm_limits=None,
    auto_norm_percentiles=(1, 99),
):
    img2d = np.asarray(img2d, dtype=np.float32)
    if threshold_space.lower() == "normalized":
        if norm_limits is None:
            vmin, vmax = quick_percentile_limits_np(
                img2d,
                p_lo=auto_norm_percentiles[0],
                p_hi=auto_norm_percentiles[1],
            )
        else:
            vmin, vmax = norm_limits
        work = np.clip((img2d - vmin) / max(vmax - vmin, 1e-6), 0, 1).astype(np.float32)
    else:
        work = img2d
    size = 2 * int(min_distance) + 1
    selem = np.ones((size, size), dtype=bool)
    clog = -ndi.gaussian_laplace(work, sigma=sigma)
    logmax = ndi.maximum_filter(clog, footprint=selem, mode="nearest")
    peak_mask = (clog == logmax)
    if threshold is None:
        step = max(1, int(min_distance))
        samp = work[::step, ::step]
        thr = np.percentile(samp, 75)
    else:
        thr = float(threshold)
    peak_mask = peak_mask & (work > thr)
    if exclude_border and min(peak_mask.shape) > 2 * int(min_distance):
        b = int(min_distance)
        peak_mask[:b, :] = False
        peak_mask[-b:, :] = False
        peak_mask[:, :b] = False
        peak_mask[:, -b:] = False
    return peak_mask.astype(bool), clog.astype(np.float32)

log("Mentor helpers defined.")
def run_mentor_v1_with_cfg(crop_row, cache_by_panel, variant_cfg, global_cfg):
    """
    Run mentor_v1-style consensus with an arbitrary config dict.
    Allows k-variant detectors to share the same logic with different k/radius.
    """
    panel_masks = []
    panel_score_maps = []
    for panel_key in global_cfg["mentor_required_panels"]:
        cache = cache_by_panel[panel_key]
        ch = cache["raw"]
        threshold = variant_cfg["threshold_561"] if panel_key.endswith("561") else variant_cfg["threshold_638"]
        mask, score_map = log_spot_mask_mentor_v1(
            ch,
            sigma=variant_cfg["sigma"],
            min_distance=variant_cfg["min_distance"],
            threshold=threshold,
            threshold_space=variant_cfg["threshold_space"],
            p_lo=global_cfg["norm_percentiles"][0],
            p_hi=global_cfg["norm_percentiles"][1],
            log_threshold=variant_cfg["log_threshold"],
            log_threshold_percentile=variant_cfg["log_threshold_percentile"],
            min_area=variant_cfg["min_area"],
        )
        panel_masks.append(mask)
        panel_score_maps.append(score_map)
    cons_mask, coords, count_img = consensus_from_masks(
        panel_masks,
        tolerance=variant_cfg["consensus_radius"],
        min_hits=variant_cfg["consensus_k"],
    )
    if len(coords) == 0:
        return coords, np.array([], dtype=np.float32), count_img, cons_mask
    scores = sample_scores(count_img, coords)
    return coords, scores, count_img, cons_mask



In [ ]:
# -------------------------------------------------------------------
# PREPROCESSING CACHE GENERATION (crop-local)
# -------------------------------------------------------------------
PREPROCESS_TIMINGS = []
CROP_PANEL_CACHE = {}
CROP_PANEL_FINGERPRINTS = {}

def compute_preprocessing_cache_for_crop(crop_row: pd.Series, chosen_panel_files: dict) -> dict:
    crop_id = crop_row["crop_id"]
    well_id = crop_row["well_id"]
    panel_cache = {}

    for panel_key, file_path in chosen_panel_files.items():
        t0 = time.perf_counter()
        try:
            full = read_image_2d(file_path)
        except Exception as _read_exc:
            log(f"[skip] crop={crop_id} panel={panel_key}: unreadable file "
                f"({type(_read_exc).__name__}: {_read_exc}) — {file_path}")
            continue
        crop = crop_image_by_registry(full, crop_row)
        crop = crop.astype(np.float32, copy=False)

        p_lo, p_hi = CFG["norm_percentiles"]
        norm_img, lo, hi = norm01(crop, p_lo, p_hi)

        cache = {
            "raw": crop,
            "norm01": norm_img,
            "norm01_ppi": norm_img,
            "white_local": white_local(norm_img, CFG["white_local_sigma_px"]),
            "peak_bg": peak_bg(norm_img, CFG["peak_bg_sigma_small_px"], CFG["peak_bg_sigma_large_px"]),
            "z_local": zscore_local(norm_img, CFG["z_local_sigma_px"]),
            "highpass": highpass(norm_img, CFG["highpass_sigma_px"]),
            "local_mad": local_mad_map(norm_img, radius=5),
            "white": white_transform(norm_img, CFG["white_local_sigma_px"]),
            "rolling_ball_residual": rolling_ball_residual(norm_img, CFG["rolling_ball_radius_px"]),
            "opening_recon_residual": opening_by_reconstruction_residual(norm_img, CFG["opening_recon_radius_px"]),
            "atrous_wavelet_bandpass": atrous_wavelet_bandpass(norm_img, CFG["wavelet_sigmas_px"]),
            "radial_symmetry_map": radial_symmetry_map(norm_img, CFG["radial_symmetry_radius_px"]),
            "hessian_blobness": hessian_blobness_map(norm_img, CFG["hessian_sigma_px"]),
            "log_normalized": normalized_log(norm_img, 2.0),
            "matched_filter": matched_filter_response(norm_img, CFG["matched_filter_sigma_px"]),
        }
        cache["image_shape_y"], cache["image_shape_x"] = crop.shape
        cache["norm_limits"] = (lo, hi)
        cache["file_path"] = file_path
        cache["panel_key"] = panel_key

        t1 = time.perf_counter()
        PREPROCESS_TIMINGS.append({
            "run_id": RUN_ID,
            "dataset_id": crop_row["dataset_id"],
            "well_id": well_id,
            "crop_id": crop_id,
            "panel_key": panel_key,
            "timing_sec": t1 - t0,
            "file_path": file_path,
        })

        panel_cache[panel_key] = cache
        CROP_PANEL_FINGERPRINTS[(crop_id, panel_key)] = sha1_file(file_path)
        log(f"[preprocess] crop={crop_id} panel={panel_key} shape={crop.shape} time={t1 - t0:.2f}s")

    return panel_cache

for _, crop_row in crop_registry.iterrows():
    crop_id = crop_row["crop_id"]
    chosen = panel_file_map[crop_id]
    log(f"Building preprocessing cache for crop {crop_id} ({crop_row['well_id']}) with {len(chosen)} panels")
    CROP_PANEL_CACHE[crop_id] = compute_preprocessing_cache_for_crop(crop_row, chosen)

preprocess_timing_df = pd.DataFrame(PREPROCESS_TIMINGS)
display(preprocess_timing_df.head())
log(f"Preprocessing cache built for {len(CROP_PANEL_CACHE)} crops")

# ── Load annotations for Spotiflow ────────────────────────────────────────
_ann_path = Path(CFG["repo_root"]) / CFG["tables_subdir"] / "annotations.parquet"
if _ann_path.exists():
    annotations_df = pd.read_parquet(_ann_path)
    _n_true = (annotations_df["label"] == "TRUE_SPOT").sum()
    log(f"Loaded annotations: {len(annotations_df)} total, {_n_true} TRUE_SPOT "
        f"across {annotations_df['crop_id'].nunique()} crops")
else:
    annotations_df = None
    log("[warn] annotations.parquet not found — Spotiflow will be skipped")


In [ ]:
# -------------------------------------------------------------------
# DETECTOR IMPLEMENTATIONS (v9 — per-method empirical thresholds)
#
# v9 principles:
#   - No adaptive noise floor (caused cross-space comparison bugs)
#   - Shape gate ONLY on rolling_ball_residual and morph_tophat
#   - Each method uses the threshold type that suits its response
#   - Mentor methods are called via frozen helpers (cell above)
# -------------------------------------------------------------------
DETECTOR_RUN_SUMMARY = []

def points_from_blob_output(blobs):
    if blobs is None or len(blobs) == 0:
        return np.zeros((0, 2), dtype=np.float32), np.array([], dtype=np.float32)
    blobs = np.asarray(blobs, dtype=np.float32)
    coords = blobs[:, :2].astype(np.float32)
    sigmas = (blobs[:, 2].astype(np.float32) if blobs.shape[1] > 2
              else np.full(len(coords), np.nan, dtype=np.float32))
    return coords, sigmas

# ── Generic local-max (quantile threshold, optional shape gate) ───────────
def detector_local_max_v9(resp, cache, *, min_distance, threshold_abs=None,
                          threshold_quantile=None, use_shape_gate=False):
    if threshold_abs is None:
        threshold_abs = robust_threshold_from_quantile(resp, threshold_quantile)
    coords = local_max_points(resp, min_distance=min_distance, threshold_abs=threshold_abs)
    scores = sample_scores(resp, coords)
    if use_shape_gate:
        coords, scores = shape_gate_filter(cache["norm01"], coords, scores, cache)
    return coords, scores

# ── scikit-image blob detectors (own absolute thresholds) ─────────────────
def detector_blob_log_v9(img, *, min_sigma, max_sigma, num_sigma, threshold):
    blobs = feature.blob_log(img, min_sigma=min_sigma, max_sigma=max_sigma,
                             num_sigma=num_sigma, threshold=threshold)
    coords, sigmas = points_from_blob_output(blobs)
    scores = sample_scores(img, coords) if len(coords) else np.array([], dtype=np.float32)
    return coords, scores, sigmas

def detector_blob_dog_v9(img, *, min_sigma, max_sigma, threshold):
    blobs = feature.blob_dog(img, min_sigma=min_sigma, max_sigma=max_sigma, threshold=threshold)
    coords, sigmas = points_from_blob_output(blobs)
    scores = sample_scores(img, coords) if len(coords) else np.array([], dtype=np.float32)
    return coords, scores, sigmas

def detector_blob_doh_v9(img, *, min_sigma, max_sigma, threshold):
    blobs = feature.blob_doh(img, min_sigma=min_sigma, max_sigma=max_sigma, threshold=threshold)
    coords, sigmas = points_from_blob_output(blobs)
    scores = sample_scores(img, coords) if len(coords) else np.array([], dtype=np.float32)
    return coords, scores, sigmas

# ── bigfish: elbow threshold + cap ────────────────────────────────────────
def detector_bigfish_v9(img_norm, *, sigma, min_distance, threshold_quantile, cap=30):
    resp = (sigma ** 2) * normalized_log(img_norm, sigma)
    size = 2 * int(min_distance) + 1
    maxf = ndi.maximum_filter(resp, size=size, mode="nearest")
    candidate_mask = (resp == maxf) & (resp > 0)
    if not np.any(candidate_mask):
        return np.zeros((0, 2), dtype=np.float32), np.array([], dtype=np.float32)
    peak_coords = np.column_stack(np.where(candidate_mask)).astype(np.float32)
    peak_vals = resp[candidate_mask].astype(np.float32)
    n = len(peak_vals)
    if n >= 10:
        sorted_vals = np.sort(peak_vals)[::-1]
        xs_n = np.linspace(0, 1, n)
        ys_n = (sorted_vals - sorted_vals[-1]) / max(sorted_vals[0] - sorted_vals[-1], 1e-8)
        dist = np.abs(xs_n + ys_n - 1.0) / np.sqrt(2)
        thr = float(sorted_vals[int(np.argmax(dist))])
    else:
        thr = robust_threshold_from_quantile(resp, threshold_quantile)
    keep = peak_vals >= thr
    coords, scores = peak_coords[keep], peak_vals[keep]
    # Cap to N strongest
    if len(coords) > cap:
        top_idx = np.argsort(scores)[-cap:]
        coords, scores = coords[top_idx], scores[top_idx]
    return coords, scores

# ── trackpy bandpass + local-max ──────────────────────────────────────────
def detector_trackpy_v9(img_norm, *, min_distance, threshold_quantile,
                        bandpass_lsig=1.5, bandpass_hsig=8.0):
    band = peak_bg(img_norm, bandpass_lsig, bandpass_hsig)
    thr = robust_threshold_from_quantile(band, threshold_quantile)
    coords = local_max_points(band, min_distance=min_distance, threshold_abs=thr)
    return coords, sample_scores(band, coords)

# ── multiscale LoG ────────────────────────────────────────────────────────
def detector_multiscale_log_v9(img_norm, *, sigmas, min_distance, threshold_quantile):
    stack = np.stack([normalized_log(img_norm, s) for s in sigmas], axis=0)
    mx = np.max(stack, axis=0)
    arg = np.argmax(stack, axis=0)
    thr = robust_threshold_from_quantile(mx, threshold_quantile)
    coords = local_max_points(mx, min_distance=min_distance, threshold_abs=thr)
    scores = sample_scores(mx, coords)
    sig_out = (np.array([sigmas[int(arg[int(round(y)), int(round(x))])]
                         for y, x in coords], dtype=np.float32)
               if len(coords) else np.array([], dtype=np.float32))
    return coords, scores, sig_out

# ── local expansion (MF-seeded weighted centroid) ─────────────────────────
def detector_local_expansion_v9(img_norm, *, min_distance, seed_quantile, expand_radius):
    resp = matched_filter_response(img_norm, CFG["matched_filter_sigma_px"])
    seed_thr = robust_threshold_from_quantile(resp, seed_quantile)
    seeds = local_max_points(resp, min_distance=min_distance, threshold_abs=seed_thr)
    coords, scores = [], []
    rad = int(expand_radius)
    for y, x in seeds:
        y0 = max(0, int(round(y)) - rad); y1 = min(resp.shape[0], int(round(y)) + rad + 1)
        x0 = max(0, int(round(x)) - rad); x1 = min(resp.shape[1], int(round(x)) + rad + 1)
        patch = resp[y0:y1, x0:x1]
        if patch.size == 0: continue
        w = np.maximum(patch - np.min(patch), 0)
        if float(w.sum()) <= 0:
            coords.append((float(y), float(x)))
            scores.append(float(resp[int(round(y)), int(round(x))]))
        else:
            yy, xx = np.indices(patch.shape)
            coords.append((y0 + float((yy*w).sum()/w.sum()),
                           x0 + float((xx*w).sum()/w.sum())))
            scores.append(float(np.max(patch)))
    return np.asarray(coords, dtype=np.float32).reshape(-1, 2), np.asarray(scores, dtype=np.float32)

# ── restrained PSF (LoG seeds → 2D Gaussian fit, no shape gate) ──────────
def _gauss2d(coords, amp, y0, x0, sy, sx, off):
    yy, xx = coords
    return (amp * np.exp(-(((yy-y0)**2)/(2*sy*sy) + ((xx-x0)**2)/(2*sx*sx))) + off).ravel()

def detector_restrained_psf_v9(img_norm, *, seed_sigma, min_distance, threshold_quantile, fit_radius):
    resp = normalized_log(img_norm, seed_sigma)
    thr = robust_threshold_from_quantile(resp, threshold_quantile)
    seeds = local_max_points(resp, min_distance=min_distance, threshold_abs=thr)
    coords, scores = [], []
    rad = int(fit_radius)
    for y, x in seeds:
        yc = int(round(y)); xc = int(round(x))
        y0 = max(0, yc-rad); y1 = min(img_norm.shape[0], yc+rad+1)
        x0 = max(0, xc-rad); x1 = min(img_norm.shape[1], xc+rad+1)
        patch = img_norm[y0:y1, x0:x1]
        if patch.shape[0] < 3 or patch.shape[1] < 3: continue
        yy, xx = np.indices(patch.shape)
        p0 = [float(patch.max()-patch.min()), float(y-y0), float(x-x0), 2.5, 2.5, float(patch.min())]
        bounds = ([0,0,0,1.5,1.5,-1],[10,patch.shape[0]-1,patch.shape[1]-1,8.0,8.0,2])
        try:
            popt, _ = curve_fit(_gauss2d, (yy,xx), patch.ravel(), p0=p0, bounds=bounds, maxfev=2000)
            amp, fy, fx, sy, sx, off = popt
            aspect = max(sy, sx) / max(min(sy, sx), 0.1)
            if aspect > 3.0:
                continue
            coords.append((y0+float(fy), x0+float(fx)))
            scores.append(float(amp / max(np.std(patch), 1e-6)))
        except Exception:
            coords.append((float(y), float(x))); scores.append(float(resp[yc, xc]))
    return np.asarray(coords, dtype=np.float32).reshape(-1, 2), np.asarray(scores, dtype=np.float32)

# ── bright rescue ─────────────────────────────────────────────────────────
def detector_bright_rescue_v9(img_raw, *, min_distance, threshold_quantile):
    thr = robust_threshold_from_quantile(img_raw, threshold_quantile)
    coords = local_max_points(img_raw, min_distance=min_distance, threshold_abs=thr)
    return coords, sample_scores(img_raw, coords)

# ── anscombe VST + LoG (Poisson-noise-aware) ──────────────────────────────
def detector_anscombe_log_v9(raw_img, *, sigma=8.0, min_distance=5, threshold_quantile=0.999):
    img = raw_img.astype(np.float32)
    img_pos = img - img.min()
    vst = (2.0 * np.sqrt(img_pos + 3.0 / 8.0)).astype(np.float32)
    clog = (-ndi.gaussian_laplace(vst, sigma=sigma)).astype(np.float32)
    thr = robust_threshold_from_quantile(clog, threshold_quantile)
    coords = local_max_points(clog, min_distance=min_distance, threshold_abs=thr)
    scores = sample_scores(clog, coords)
    return coords, scores

# ── local SNR detector ────────────────────────────────────────────────────
def detector_local_snr_v9(peak_bg_img, local_mad_img, *, min_distance=5, snr_min=8.0):
    signal = np.clip(peak_bg_img.astype(np.float32), 0, None)
    noise = np.clip(local_mad_img.astype(np.float32), 1e-7, None)
    snr_map = (signal / noise).astype(np.float32)
    coords = local_max_points(snr_map, min_distance=min_distance, threshold_abs=snr_min)
    scores = sample_scores(snr_map, coords)
    return coords, scores

# ── Single-panel dispatcher ───────────────────────────────────────────────
def run_single_panel_detector(method_id, cache, params):
    raw      = cache["raw"]
    norm_img = cache["norm01"]
    white_img= cache["white"]
    peakbg   = cache["peak_bg"]
    zloc     = cache["z_local"]
    atrous   = cache["atrous_wavelet_bandpass"]
    radial   = cache["radial_symmetry_map"]
    hess     = cache["hessian_blobness"]
    roll     = cache["rolling_ball_residual"]
    openr    = cache["opening_recon_residual"]
    mf       = cache["matched_filter"]
    use_sg   = params.get("use_shape_gate", False)

    if method_id == "proj_local_max_raw":
        return detector_local_max_v9(raw, cache, min_distance=params["min_distance"],
                   threshold_quantile=params["threshold_quantile"]), "raw"
    if method_id == "proj_log_norm":
        resp = normalized_log(norm_img, params["sigma"])
        return (detector_local_max_v9(resp, cache, min_distance=params["min_distance"],
                    threshold_quantile=params["threshold_quantile"]), "log_normalized")
    if method_id == "proj_log_white":
        resp = normalized_log(white_img, params["sigma"])
        return (detector_local_max_v9(resp, cache, min_distance=params["min_distance"],
                    threshold_quantile=params["threshold_quantile"]), "white")
    if method_id == "proj_peakbg_max":
        return detector_local_max_v9(peakbg, cache, min_distance=params["min_distance"],
                   threshold_quantile=params["threshold_quantile"]), "peak_bg"
    if method_id == "proj_zlocal_max":
        return (detector_local_max_v9(zloc, cache, min_distance=params["min_distance"],
                    threshold_abs=params["threshold_abs"]), "z_local")
    if method_id == "morph_tophat":
        resp = morphology.white_tophat(norm_img, morphology.disk(5)).astype(np.float32)
        return (detector_local_max_v9(resp, cache, min_distance=params["min_distance"],
                    threshold_quantile=params["threshold_quantile"], use_shape_gate=True), "morph_tophat")
    if method_id == "rolling_ball_residual":
        return detector_local_max_v9(roll, cache, min_distance=params["min_distance"],
                   threshold_quantile=params["threshold_quantile"], use_shape_gate=True), "rolling_ball_residual"
    if method_id == "opening_recon_residual":
        return detector_local_max_v9(openr, cache, min_distance=params["min_distance"],
                   threshold_quantile=params["threshold_quantile"]), "opening_recon_residual"
    if method_id == "atrous_wavelet":
        return detector_local_max_v9(atrous, cache, min_distance=params["min_distance"],
                   threshold_quantile=params["threshold_quantile"]), "atrous_wavelet_bandpass"
    if method_id == "radial_symmetry":
        return detector_local_max_v9(radial, cache, min_distance=params["min_distance"],
                   threshold_quantile=params["threshold_quantile"]), "radial_symmetry_map"
    if method_id == "hessian_blobness":
        return detector_local_max_v9(hess, cache, min_distance=params["min_distance"],
                   threshold_quantile=params["threshold_quantile"]), "hessian_blobness"
    if method_id == "matched_filter_v2":
        return detector_local_max_v9(mf, cache, min_distance=params["min_distance"],
                   threshold_quantile=params["threshold_quantile"]), "matched_filter"
    if method_id == "local_expansion":
        return detector_local_expansion_v9(norm_img, **params), "local_expansion"
    if method_id == "restrained_psf":
        return detector_restrained_psf_v9(norm_img, **params), "restrained_psf"
    if method_id == "bright_rescue":
        return detector_bright_rescue_v9(raw, **params), "bright_rescue"
    if method_id == "bigfish_style":
        return detector_bigfish_v9(norm_img, **params), "bigfish_style"
    if method_id == "trackpy":
        return detector_trackpy_v9(norm_img, **params), "trackpy"
    if method_id == "sk_log":
        c, s, sig = detector_blob_log_v9(norm_img, **params)
        return (c, s, sig), "sk_log"
    if method_id == "sk_dog":
        c, s, sig = detector_blob_dog_v9(norm_img, **params)
        return (c, s, sig), "sk_dog"
    if method_id == "sk_doh":
        c, s, sig = detector_blob_doh_v9(norm_img, **params)
        return (c, s, sig), "sk_doh"
    if method_id == "multiscale_log":
        c, s, sig = detector_multiscale_log_v9(norm_img, **params)
        return (c, s, sig), "multiscale_log"
    if method_id == "anscombe_log":
        return detector_anscombe_log_v9(raw, **params), "anscombe_log"
    if method_id == "local_snr":
        return detector_local_snr_v9(peakbg, cache["local_mad"], **params), "local_snr"
    raise KeyError(method_id)

# ── Mentor runners (FROZEN) ───────────────────────────────────────────────
def run_mentor_v1(crop_row, cache_by_panel):
    cfg = CFG["mentor_v1"]
    panel_masks = []
    for panel_key in CFG["mentor_required_panels"]:
        cache = cache_by_panel[panel_key]
        threshold = cfg["threshold_561"] if panel_key.endswith("561") else cfg["threshold_638"]
        mask, _ = log_spot_mask_mentor_v1(
            cache["raw"], sigma=cfg["sigma"], min_distance=cfg["min_distance"],
            threshold=threshold, threshold_space=cfg["threshold_space"],
            p_lo=CFG["norm_percentiles"][0], p_hi=CFG["norm_percentiles"][1],
            log_threshold=cfg["log_threshold"],
            log_threshold_percentile=cfg["log_threshold_percentile"],
            min_area=cfg["min_area"],
        )
        panel_masks.append(mask)
    cons_mask, coords, count_img = consensus_from_masks(
        panel_masks, tolerance=cfg["consensus_radius"], min_hits=cfg["consensus_k"])
    if len(coords) == 0:
        return coords, np.array([], dtype=np.float32), count_img, cons_mask
    return coords, sample_scores(count_img, coords), count_img, cons_mask

def run_mentor_v2(crop_row, cache_by_panel):
    cfg = CFG["mentor_v2"]
    panel_masks = []
    for panel_key in CFG["mentor_required_panels"]:
        cache = cache_by_panel[panel_key]
        threshold = cfg["threshold_561"] if panel_key.endswith("561") else cfg["threshold_638"]
        norm_limits = cache["norm_limits"] if cfg["threshold_space"] == "normalized" else None
        mask, _ = log_peaks_2d_mentor_v2(
            cache["raw"], sigma=cfg["sigma"], min_distance=cfg["min_distance"],
            threshold=threshold, exclude_border=cfg["exclude_border"],
            threshold_space=cfg["threshold_space"], norm_limits=norm_limits,
            auto_norm_percentiles=cfg["auto_norm_percentiles"],
        )
        panel_masks.append(mask)
    cons_mask, coords, count_img = consensus_from_masks(
        panel_masks, tolerance=cfg["consensus_radius"], min_hits=cfg["consensus_k"])
    if len(coords) == 0:
        return coords, np.array([], dtype=np.float32), count_img, cons_mask
    return coords, sample_scores(count_img, coords), count_img, cons_mask

def run_consensus_v2(crop_row, cache_by_panel):
    cfg = CFG["detector_defaults"]["consensus_v2"]
    prim_thr_q = cfg.get("primitive_threshold_quantile", 0.9993)
    panel_masks = []
    for panel_key in CFG["mentor_required_panels"]:
        cache = cache_by_panel[panel_key]
        primitive_masks = []
        resp1 = normalized_log(cache["norm01"], 8.0)
        primitive_masks.append(resp1 >= robust_threshold_from_quantile(resp1, prim_thr_q))
        resp2 = cache["matched_filter"]
        primitive_masks.append(resp2 >= robust_threshold_from_quantile(resp2, prim_thr_q))
        resp3 = cache["radial_symmetry_map"]
        primitive_masks.append(resp3 >= robust_threshold_from_quantile(resp3, prim_thr_q))
        panel_mask, _, _ = consensus_from_masks(
            primitive_masks, tolerance=cfg["panel_consensus_radius"],
            min_hits=cfg["panel_consensus_k"])
        panel_masks.append(panel_mask)
    cons_mask, coords, count_img = consensus_from_masks(
        panel_masks, tolerance=cfg["global_consensus_radius"],
        min_hits=cfg["global_consensus_k"])
    return coords, sample_scores(count_img, coords), count_img, cons_mask

DETECTOR_FAMILY = {
    "mentor_v1": "mentor", "mentor_v2": "mentor", "consensus_v2": "aggregator",
    "mentor_v1_k2": "mentor", "mentor_v1_k4": "mentor", "mentor_v1_tight": "mentor",
    "matched_filter_v2": "matched_filter", "local_expansion": "expansion",
    "restrained_psf": "psf_fit", "bright_rescue": "rescue",
    "sk_log": "blob_skimage", "sk_dog": "blob_skimage", "sk_doh": "blob_skimage",
    "bigfish_style": "log_localmax", "trackpy": "bandpass_localmax",
    "proj_local_max_raw": "localmax",
    "proj_log_norm": "log", "proj_log_white": "log",
    "proj_peakbg_max": "difference", "proj_zlocal_max": "zscore",
    "atrous_wavelet": "wavelet", "morph_tophat": "morphology",
    "multiscale_log": "log_multiscale", "radial_symmetry": "radial_symmetry",
    "hessian_blobness": "hessian",
    "rolling_ball_residual": "background_residual",
    "opening_recon_residual": "background_residual",
    "anscombe_log": "log_poisson_aware",
    "local_snr": "snr",
    "spotiflow_finetuned": "deep_learning",
}

log("Detector implementations defined (v9).")


In [ ]:
# ---------------------------------------------------------------------------
# APPEND-C  Spotiflow — fine-tune on NB02 annotations + full-crop inference
#
# Spotiflow (Dominguez Mantes et al., Nature Methods 2025, 22:1495-1504)
# formulates spot detection as joint multiscale heatmap regression +
# stereographic flow regression, yielding subpixel-accurate positions
# that are threshold-agnostic at inference time.
#
# WHY THIS ADDS VALUE over classical detectors:
# - Threshold-agnostic: no per-channel sensitivity tuning needed.
# - Subpixel accuracy: stereographic flow gives <0.1 px localisation error.
# - Better recall at SNR~2-3 than LoG, DoG, WMP, or h-dome per the
#   Nature Methods benchmark (Extended Data Fig. 3-4).
# - Generalises across channels: one model handles 561 and 638 nm.
#
# PSEUDO-GROUND-TRUTH SOURCE:
# NB02 annotates TRUE_SPOT points on crop images using the 4-panel
# interactive viewer.  Coordinates are stored as refined_click_crop_y/x_px
# (local-maximum snapped within 5px of the raw click) in annotations.parquet.
# These annotations are the highest-quality labels in the project — human
# expert clicks on real images — making them ideal for Spotiflow fine-tuning.
#
# TRAINING PROTOCOL:
# - Fine-tune from "general" pretrained model (trained on diverse 2D FISH /
#   spatial transcriptomics data with >1 M annotated spots).
# - 80/20 crop-level train/val split (no spot-level leakage between splits).
# - Training image: norm01 of C1_638 panel (primary barcode channel).
# - 50 epochs, lr=3e-4, crop_size=256, smart_crop=True (preferentially
#   crops around annotated spots during training augmentation).
# - Even 3 annotated crops give F1 0.77->0.92 per Spotiflow paper.
#
# INFERENCE:
# - Fine-tuned model is run on every barcode panel of every crop.
# - Outputs: subpixel (y,x) coordinates + heatmap probability score.
# - Saved to candidate_raw with method_id="spotiflow_finetuned",
#   score_is_calibrated=True (heatmap output is probability-like).
# ---------------------------------------------------------------------------

_APPEND_C_RAW_DFS = []
spotiflow_model_trained = None

try:
    from spotiflow.model import Spotiflow, SpotiflowTrainingConfig
    HAS_SPOTIFLOW = True
    log("spotiflow imported OK")
except ImportError:
    HAS_SPOTIFLOW = False
    log("[warn] spotiflow not installed — pip install spotiflow")

if HAS_SPOTIFLOW and annotations_df is not None:

    SPOTIFLOW_DIR   = REPO_ROOT / "artifacts" / "spotiflow_finetune"
    SPOTIFLOW_MODEL = SPOTIFLOW_DIR / "model"
    for _d in [SPOTIFLOW_DIR, SPOTIFLOW_MODEL]:
        _d.mkdir(parents=True, exist_ok=True)

    # ---- Build (image, spots) pairs from NB02 annotations -----------------
    true_ann = annotations_df[annotations_df["label"] == "TRUE_SPOT"].copy()
    log(f"TRUE_SPOT annotations: {len(true_ann)} across "
        f"{true_ann['crop_id'].nunique()} crops")

    # Spotiflow expects images as 2D float32 numpy arrays.
    # It expects spot coordinates as (N, 2) float32 in (row=y, col=x) order,
    # in crop-local pixel units — exactly what refined_click_crop_y/x_px gives.
    import random as _random
    _random.seed(42)
    all_cids = true_ann["crop_id"].unique().tolist()
    _random.shuffle(all_cids)
    _n_val = max(1, int(len(all_cids) * 0.2))
    val_cids   = set(all_cids[:_n_val])
    train_cids = set(all_cids[_n_val:])
    log(f"Train crops: {len(train_cids)}, Val crops: {len(val_cids)}")

    train_imgs, train_spots = [], []
    val_imgs,   val_spots   = [], []

    # Panel preference order — train on primary barcode channel
    _panel_pref = ["C1_638", "C4_638", "C5_638", "C4_561", "C5_561"]

    for cid in all_cids:
        pmap = CROP_PANEL_CACHE.get(cid, {})
        panel = next((p for p in _panel_pref if p in pmap), None)
        if panel is None:
            log(f"  [skip spotiflow] crop {cid[-12:]}: no barcode panel in cache")
            continue

        img = pmap[panel]["norm01"].astype(np.float32)
        H, W = img.shape

        crop_ann = true_ann[true_ann["crop_id"] == cid]
        spots = crop_ann[["refined_click_crop_y_px",
                           "refined_click_crop_x_px"]].values.astype(np.float32)
        # Clamp to image bounds (defensive)
        spots[:, 0] = np.clip(spots[:, 0], 0.0, float(H - 1))
        spots[:, 1] = np.clip(spots[:, 1], 0.0, float(W - 1))

        if len(spots) == 0:
            continue

        if cid in train_cids:
            train_imgs.append(img);  train_spots.append(spots)
        else:
            val_imgs.append(img);    val_spots.append(spots)

    log(f"Spotiflow training: {len(train_imgs)} images, "
        f"{sum(len(s) for s in train_spots)} spots")
    log(f"Spotiflow val:      {len(val_imgs)} images, "
        f"{sum(len(s) for s in val_spots)} spots")

    if len(train_imgs) == 0:
        log("[warn] No training pairs built — skipping fine-tuning")
        HAS_SPOTIFLOW = False
    else:
        # ---- Fine-tune ------------------------------------------------------
        log("Loading Spotiflow 'general' pretrained model ...")
        spotiflow_model_trained = Spotiflow.from_pretrained(
            "general",
            inference_mode=False,
        )

        # Use val data if available; otherwise reuse last train image
        _v_imgs  = val_imgs  if val_imgs  else [train_imgs[-1]]
        _v_spots = val_spots if val_spots else [train_spots[-1]]

        _t_ft = time.perf_counter()
        log("Fine-tuning started ...")

        spotiflow_model_trained.fit(
            train_imgs,
            train_spots,
            _v_imgs,
            _v_spots,
            save_dir=str(SPOTIFLOW_MODEL),
            train_params=SpotiflowTrainingConfig(
                num_epochs=50,
                learning_rate=3e-4,
                batch_size=4,
                crop_size=256,
                smart_crop=True,     # bias augmentation crops towards spots
                augment_flips=True,
                augment_rotations=True,
            ),
        )
        log(f"Fine-tuning done in {time.perf_counter()-_t_ft:.0f}s")
        log(f"Model saved → {SPOTIFLOW_MODEL}")

        # Reload in inference mode
        spotiflow_model_trained = Spotiflow.from_folder(
            str(SPOTIFLOW_MODEL), inference_mode=True
        )

        # ---- Inference on all crops ----------------------------------------
        log("Running inference on all crops ...")
        _t_inf_total = time.perf_counter()

        for _, crop_row in crop_registry.iterrows():
            crop_id    = crop_row["crop_id"]
            well_id    = crop_row["well_id"]
            dataset_id = crop_row["dataset_id"]
            oy = int(crop_row["well_ymin_px"])
            ox = int(crop_row["well_xmin_px"])
            pmap = CROP_PANEL_CACHE.get(crop_id, {})

            for pk in sorted(k for k in pmap if re.match(r"^C[145]_(561|638)$", k)):
                img    = pmap[pk]["norm01"].astype(np.float32)
                H, W   = img.shape
                src_fp = CROP_PANEL_FINGERPRINTS.get((crop_id, pk), "recovered")

                _t_inf = time.perf_counter()
                try:
                    spots, details = spotiflow_model_trained.predict(
                        img,
                        min_distance=3,  # v8: raised from 1 to match physical spot scale
                        peak_mode="fast",  # "accurate" is slower but ~+0.5% F1
                    )
                except Exception as exc:
                    log(f"  [error spotiflow] {pk}: {exc}")
                    continue

                if len(spots) == 0:
                    continue

                # details.intens: per-spot heatmap probability (calibrated)
                scores = (details.intens.astype(np.float32)
                          if hasattr(details, "intens")
                          else np.ones(len(spots), dtype=np.float32))

                df = make_detection_df(
                    spots, scores,
                    run_id=RUN_ID,
                    dataset_id=dataset_id,
                    well_id=well_id,
                    crop_id=crop_id,
                    method_id="spotiflow_finetuned",
                    method_family="deep_learning",
                    source_view_id=f"{pk}__spotiflow_finetuned",
                    score_type="heatmap_probability",
                    score_is_calibrated=True,
                    sigma_px=None,
                    timing_sec=time.perf_counter() - _t_inf,
                    crop_origin_well_y_px=oy,
                    crop_origin_well_x_px=ox,
                    image_shape_y=H,
                    image_shape_x=W,
                    coord_semantics="fit_centroid",  # subpixel flow-refined
                    source_image_fingerprint=src_fp,
                )
                _APPEND_C_RAW_DFS.append(df)

        log(f"Inference done in {time.perf_counter()-_t_inf_total:.1f}s")

elif HAS_SPOTIFLOW and annotations_df is None:
    log("[warn] Spotiflow available but no annotations found — run Notebook 02 first")

spotiflow_append_df = (pd.concat(_APPEND_C_RAW_DFS, ignore_index=True)
                       if _APPEND_C_RAW_DFS else pd.DataFrame())
log(f"Append-C: {len(spotiflow_append_df)} Spotiflow detections")


In [ ]:
# -------------------------------------------------------------------
# DETECTOR EXECUTION (v9 — all methods in one unified loop)
# -------------------------------------------------------------------
import sys

RAW_DETECTIONS = []

# All single-panel classical detectors
_SINGLE_PANEL_IDS = [
    "matched_filter_v2", "local_expansion", "restrained_psf", "bright_rescue",
    "sk_log", "sk_dog", "sk_doh", "bigfish_style", "trackpy",
    "proj_local_max_raw", "proj_log_norm", "proj_log_white",
    "proj_peakbg_max", "proj_zlocal_max", "atrous_wavelet", "morph_tophat",
    "multiscale_log", "radial_symmetry", "hessian_blobness",
    "rolling_ball_residual", "opening_recon_residual",
    "anscombe_log", "local_snr",
]

_MENTOR_IDS = [
    "mentor_v1", "mentor_v2", "consensus_v2",
    "mentor_v1_k2", "mentor_v1_k4", "mentor_v1_tight",
]

detector_ids = _MENTOR_IDS + _SINGLE_PANEL_IDS

def add_summary(crop_id, well_id, method_id, source_view_id, n_pts, timing_sec, status):
    DETECTOR_RUN_SUMMARY.append({
        "run_id": RUN_ID, "crop_id": crop_id, "well_id": well_id,
        "method_id": method_id, "source_view_id": source_view_id,
        "n_points": int(n_pts), "timing_sec": float(timing_sec), "status": status,
    })

# ── Progress tracking ─────────────────────────────────────────────────────
n_crops = len(crop_registry)
t_global_start = time.perf_counter()
task_counter = 0
est_total = n_crops * (len(_MENTOR_IDS) + len(_SINGLE_PANEL_IDS) * 5)

def progress_bar(current, total, prefix=""):
    frac = current / max(total, 1)
    filled = int(40 * frac)
    bar = "\u2588" * filled + "\u2591" * (40 - filled)
    elapsed = time.perf_counter() - t_global_start
    eta = (elapsed / max(current, 1)) * (total - current)
    eta_str = f"{eta:.0f}s" if eta < 3600 else f"{eta/60:.1f}m"
    sys.stdout.write(f"\r{prefix} |{bar}| {current}/{total} ({frac*100:.1f}%) ETA {eta_str}  ")
    sys.stdout.flush()

for crop_idx, (_, crop_row) in enumerate(crop_registry.iterrows()):
    crop_id = crop_row["crop_id"]
    well_id = crop_row["well_id"]
    dataset_id = crop_row["dataset_id"]
    crop_origin_y = int(crop_row["well_ymin_px"])
    crop_origin_x = int(crop_row["well_xmin_px"])
    cache_by_panel = CROP_PANEL_CACHE[crop_id]

    log("=" * 90)
    log(f"START crop={crop_id} well={well_id} [{crop_idx+1}/{n_crops}]")

    crop_raw_dfs = []

    # ── Mentor / consensus methods ─────────────────────────────────────────
    for method_id in _MENTOR_IDS:
        MENTOR_VARIANT_CFGS = {
            "mentor_v1": CFG["mentor_v1"], "mentor_v2": CFG["mentor_v2"],
            "mentor_v1_k2": CFG["mentor_v1_k2"], "mentor_v1_k4": CFG["mentor_v1_k4"],
            "mentor_v1_tight": CFG["mentor_v1_tight"],
        }
        if not all(p in cache_by_panel for p in CFG["mentor_required_panels"]):
            add_summary(crop_id, well_id, method_id, "mentor", 0, 0.0, "skipped_missing_panels")
            continue

        t0 = time.perf_counter()
        if method_id == "mentor_v2":
            coords, scores, score_map, cons_mask = run_mentor_v2(crop_row, cache_by_panel)
            source_view_id = "mentor_barcode_consensus_v2"
        elif method_id == "consensus_v2":
            coords, scores, score_map, cons_mask = run_consensus_v2(crop_row, cache_by_panel)
            source_view_id = "consensus_v2_barcode5"
        else:
            variant_cfg = MENTOR_VARIANT_CFGS[method_id]
            coords, scores, score_map, cons_mask = run_mentor_v1_with_cfg(
                crop_row, cache_by_panel, variant_cfg, CFG)
            source_view_id = f"mentor_barcode_consensus_{method_id}"
        t1 = time.perf_counter()

        source_fp = sha1_text("|".join(CROP_PANEL_FINGERPRINTS[(crop_id, p)]
                                        for p in CFG["mentor_required_panels"]))
        df = make_detection_df(
            coords, scores, run_id=RUN_ID, dataset_id=dataset_id,
            well_id=well_id, crop_id=crop_id, method_id=method_id,
            method_family=DETECTOR_FAMILY[method_id],
            source_view_id=source_view_id, score_type="panel_vote_count",
            score_is_calibrated=False,
            sigma_px=MENTOR_VARIANT_CFGS.get(method_id, {}).get("sigma"),
            timing_sec=t1-t0, crop_origin_well_y_px=crop_origin_y,
            crop_origin_well_x_px=crop_origin_x,
            image_shape_y=cache_by_panel[CFG["mentor_required_panels"][0]]["image_shape_y"],
            image_shape_x=cache_by_panel[CFG["mentor_required_panels"][0]]["image_shape_x"],
            coord_semantics="medoid_like", source_image_fingerprint=source_fp,
        )
        crop_raw_dfs.append(df)
        add_summary(crop_id, well_id, method_id, source_view_id, len(df), t1-t0, "ok")
        log(f"[done] {method_id} n={len(df)} t={t1-t0:.2f}s")
        task_counter += 1
        progress_bar(task_counter, est_total, "Detectors")

    # ── Single-panel methods ───────────────────────────────────────────────
    for method_id in _SINGLE_PANEL_IDS:
        for panel_key in sorted(k for k in cache_by_panel if re.match(r'^C[145]_(561|638)$', k)):
            cache = cache_by_panel[panel_key]
            t0 = time.perf_counter()
            try:
                out, semantic_view = run_single_panel_detector(
                    method_id, cache, CFG["detector_defaults"][method_id])
                if len(out) == 2:
                    coords, scores = out
                    sigma_arr = None
                else:
                    coords, scores, sigma_arr = out
                t1 = time.perf_counter()
                source_fp = CROP_PANEL_FINGERPRINTS[(crop_id, panel_key)]
                sig_use = None
                if sigma_arr is not None and len(sigma_arr):
                    sig_use = float(np.nanmedian(sigma_arr))
                elif "sigma" in CFG["detector_defaults"][method_id]:
                    sig_use = CFG["detector_defaults"][method_id]["sigma"]
                df = make_detection_df(
                    coords, scores, run_id=RUN_ID, dataset_id=dataset_id,
                    well_id=well_id, crop_id=crop_id, method_id=method_id,
                    method_family=DETECTOR_FAMILY[method_id],
                    source_view_id=f"{panel_key}__{semantic_view}",
                    score_type="response_value", score_is_calibrated=False,
                    sigma_px=sig_use, timing_sec=t1-t0,
                    crop_origin_well_y_px=crop_origin_y,
                    crop_origin_well_x_px=crop_origin_x,
                    image_shape_y=cache["image_shape_y"],
                    image_shape_x=cache["image_shape_x"],
                    coord_semantics="local_maximum" if method_id not in ("restrained_psf",) else "fit_centroid",
                    source_image_fingerprint=source_fp,
                )
                crop_raw_dfs.append(df)
                add_summary(crop_id, well_id, method_id, f"{panel_key}__{semantic_view}",
                           len(df), t1-t0, "ok")
            except Exception as e:
                t1 = time.perf_counter()
                add_summary(crop_id, well_id, method_id, panel_key, 0, t1-t0,
                           f"error:{type(e).__name__}")
                log(f"[error] {method_id} {panel_key}: {type(e).__name__}: {e}")
            task_counter += 1
            progress_bar(task_counter, est_total, "Detectors")

    # ── Spotiflow (if available) ───────────────────────────────────────────
    if 'spotiflow_model_trained' in dir() and spotiflow_model_trained is not None:
        for panel_key in sorted(k for k in cache_by_panel if re.match(r'^C[145]_(561|638)$', k)):
            cache = cache_by_panel[panel_key]
            t0 = time.perf_counter()
            try:
                img = cache["norm01"].astype(np.float32)
                spots, details = spotiflow_model_trained.predict(
                    img, min_distance=3, peak_mode="fast")
                if len(spots) > 0:
                    scores = (details.intens.astype(np.float32)
                              if hasattr(details, "intens")
                              else np.ones(len(spots), dtype=np.float32))
                    df = make_detection_df(
                        spots, scores, run_id=RUN_ID, dataset_id=dataset_id,
                        well_id=well_id, crop_id=crop_id,
                        method_id="spotiflow_finetuned",
                        method_family="deep_learning",
                        source_view_id=f"{panel_key}__spotiflow_finetuned",
                        score_type="heatmap_probability",
                        score_is_calibrated=True, sigma_px=None,
                        timing_sec=time.perf_counter()-t0,
                        crop_origin_well_y_px=crop_origin_y,
                        crop_origin_well_x_px=crop_origin_x,
                        image_shape_y=cache["image_shape_y"],
                        image_shape_x=cache["image_shape_x"],
                        coord_semantics="fit_centroid",
                        source_image_fingerprint=CROP_PANEL_FINGERPRINTS.get(
                            (crop_id, panel_key), "recovered"),
                    )
                    crop_raw_dfs.append(df)
                    add_summary(crop_id, well_id, "spotiflow_finetuned",
                               f"{panel_key}__spotiflow", len(df),
                               time.perf_counter()-t0, "ok")
            except Exception as e:
                add_summary(crop_id, well_id, "spotiflow_finetuned", panel_key,
                           0, time.perf_counter()-t0, f"error:{type(e).__name__}")

    # ── Per-crop budget enforcement ────────────────────────────────────────
    n_before = sum(len(df) for df in crop_raw_dfs)
    crop_raw_dfs = apply_per_method_budget(crop_raw_dfs, crop_id)
    n_after = sum(len(df) for df in crop_raw_dfs)
    RAW_DETECTIONS.extend(crop_raw_dfs)
    log(f"  crop {crop_id[-12:]} total: {n_before} raw -> {n_after} after budget")

print()  # newline after progress bar
candidate_raw = pd.concat(RAW_DETECTIONS, ignore_index=True) if RAW_DETECTIONS else pd.DataFrame()
detector_summary_df = pd.DataFrame(DETECTOR_RUN_SUMMARY)

log(f"Total raw detections = {len(candidate_raw)}")

# ── Per-method summary ────────────────────────────────────────────────────
log("Per-method detection counts:")
display(
    detector_summary_df[detector_summary_df["status"] == "ok"]
    .groupby("method_id")["n_points"]
    .agg(["sum", "mean", "median", "max", "min", "count"])
    .rename(columns={"sum": "total", "mean": "avg", "count": "n_runs"})
    .sort_values("total", ascending=False)
)


In [ ]:
# -------------------------------------------------------------------
# CANDIDATE UNION (deterministic radius-graph merge)
# -------------------------------------------------------------------
def connected_components_radius(coords_well_yx: np.ndarray, radius: float):
    n = len(coords_well_yx)
    if n == 0:
        return np.array([], dtype=int)
    if n == 1:
        return np.array([0], dtype=int)
    tree = cKDTree(coords_well_yx)
    pairs = list(tree.query_pairs(r=radius))
    if not pairs:
        return np.arange(n, dtype=int)
    rows = []
    cols = []
    for i, j in pairs:
        rows.extend([i, j])
        cols.extend([j, i])
    data = np.ones(len(rows), dtype=np.uint8)
    graph = coo_matrix((data, (rows, cols)), shape=(n, n))
    n_comp, labels = connected_components(graph, directed=False)
    isolated = np.setdiff1d(np.arange(n), np.unique(np.r_[rows, cols]))
    for i in isolated:
        labels[i] = labels.max() + 1
    _, labels2 = np.unique(labels, return_inverse=True)
    return labels2.astype(int)

def deterministic_union_id(member_ids):
    canon = "|".join(sorted(member_ids))
    return sha1_text(canon)

def deterministic_cluster_id(union_ids):
    canon = "|".join(sorted(union_ids))
    return sha1_text("cluster|" + canon)

def build_union_tables(candidate_raw: pd.DataFrame, merge_radius_px: float):
    if len(candidate_raw) == 0:
        return (pd.DataFrame(), pd.DataFrame())
    union_rows = []
    membership_rows = []

    group_cols = ["dataset_id", "well_id", "crop_id"]
    groups = list(candidate_raw.groupby(group_cols, sort=True))
    n_groups = len(groups)
    n_methods = candidate_raw["method_id"].nunique()
    t_total_start = time.perf_counter()

    for g_idx, (keys, g) in enumerate(groups):
        dataset_id_g, well_id_g, crop_id_g = keys
        t_group = time.perf_counter()
        g = g.sort_values(["method_id", "source_view_id", "raw_detection_id"]).reset_index(drop=True)
        coords = g[["well_y_px", "well_x_px"]].to_numpy(dtype=np.float32)
        labels = connected_components_radius(coords, merge_radius_px)
        unique_comps = np.unique(labels)
        comp_sizes   = np.bincount(labels)
        n_large      = int((comp_sizes > 10).sum())

        for comp_id in unique_comps:
            idx = np.flatnonzero(labels == comp_id)
            comp = g.iloc[idx].copy().sort_values("raw_detection_id").reset_index(drop=True)
            member_ids = comp["raw_detection_id"].tolist()
            union_id = deterministic_union_id(member_ids)

            comp_coords = comp[["well_y_px", "well_x_px"]].to_numpy(dtype=np.float32)
            centroid = comp_coords.mean(axis=0)
            # O(N) medoid: point closest to centroid.
            # Exact for convex/round clusters (all real spot clusters).
            # Avoids O(N^2) pairwise matrix that causes MemoryError on
            # over-detected crops.
            dists_to_centroid = np.sqrt(((comp_coords - centroid) ** 2).sum(axis=1))
            medoid_idx = int(np.argmin(dists_to_centroid))
            medoid = comp_coords[medoid_idx]
            d_med  = np.sqrt(((comp_coords - medoid) ** 2).sum(axis=1))
            top = comp.sort_values(
                ["detection_score_norm", "detection_score_raw", "raw_detection_id"],
                ascending=[False, False, True],
            ).iloc[0]

            proposer_set = sorted(comp["method_id"].astype(str).unique().tolist())
            family_set = sorted(comp["method_family"].astype(str).unique().tolist())

            union_rows.append({
                "union_id": union_id,
                "run_id": top["run_id"],
                "dataset_id": top["dataset_id"],
                "well_id": top["well_id"],
                "crop_id": top["crop_id"],
                "union_n_members": int(len(comp)),
                "union_centroid_well_y_px": float(centroid[0]),
                "union_centroid_well_x_px": float(centroid[1]),
                "union_medoid_well_y_px": float(medoid[0]),
                "union_medoid_well_x_px": float(medoid[1]),
                "union_radius_px": float(np.max(dists_to_centroid) if len(dists_to_centroid) else 0.0),
                "union_bbox_ymin_px": float(np.min(comp_coords[:, 0])),
                "union_bbox_xmin_px": float(np.min(comp_coords[:, 1])),
                "union_bbox_ymax_px": float(np.max(comp_coords[:, 0])),
                "union_bbox_xmax_px": float(np.max(comp_coords[:, 1])),
                "top_method_id": str(top["method_id"]),
                "top_method_score_raw": float(top["detection_score_raw"]),
                "top_method_score_norm": float(top["detection_score_norm"]),
                "proposer_support_count": int(len(proposer_set)),
                "proposer_support_fraction": float(len(proposer_set) / max(n_methods, 1)),
                "proposer_set_canonical": "|".join(proposer_set),
                "family_support_count": int(len(family_set)),
                "family_set_canonical": "|".join(family_set),
                "cluster_id": None,
                **{k: top[k] for k in BASE_PROVENANCE.keys()},
            })

            ranked = comp.sort_values(
                ["detection_score_norm", "detection_score_raw", "raw_detection_id"],
                ascending=[False, False, True],
            ).reset_index(drop=True)
            # Vectorised membership — avoids per-row Python overhead
            yx_ranked  = ranked[["well_y_px", "well_x_px"]].to_numpy(dtype=np.float32)
            d_to_cent  = np.sqrt(((yx_ranked - centroid) ** 2).sum(axis=1))
            d_to_med   = np.sqrt(((yx_ranked - medoid)   ** 2).sum(axis=1))
            medoid_rid = comp.iloc[medoid_idx]["raw_detection_id"]
            for rank in range(len(ranked)):
                membership_rows.append({
                    "union_id":                       union_id,
                    "raw_detection_id":               ranked.iloc[rank]["raw_detection_id"],
                    "member_rank_by_score":           rank + 1,
                    "member_is_medoid":               bool(ranked.iloc[rank]["raw_detection_id"] == medoid_rid),
                    "member_distance_to_centroid_px": float(d_to_cent[rank]),
                    "member_distance_to_medoid_px":   float(d_to_med[rank]),
                })
        # ── per-crop progress ──────────────────────────────────────────────
        t_elapsed = time.perf_counter() - t_group
        t_total   = time.perf_counter() - t_total_start
        pct = (g_idx + 1) / n_groups * 100
        eta = (t_total / (g_idx + 1)) * (n_groups - g_idx - 1)
        log(f"  [{g_idx+1:2d}/{n_groups}] {well_id_g} {crop_id_g[-12:]} | "
            f"raw={len(g):6d} comps={len(unique_comps):6d} "
            f"large(>10)={n_large:4d} max={int(comp_sizes.max()):6d} | "
            f"{t_elapsed:.1f}s | {pct:.0f}% ETA {eta:.0f}s")

    log(f"Union loop done in {time.perf_counter()-t_total_start:.1f}s")
    return pd.DataFrame(union_rows), pd.DataFrame(membership_rows)

t0 = time.perf_counter()
log(f"Building union (merge_radius={CFG['merge_radius_px']}px, "
    f"{len(candidate_raw)} raw detections) ...")
candidate_union, candidate_union_membership = build_union_tables(candidate_raw, CFG["merge_radius_px"])
t1 = time.perf_counter()
log(f"Union built: n_union={len(candidate_union)} n_membership={len(candidate_union_membership)} time={t1 - t0:.2f}s")

display(candidate_union.head())
display(candidate_union_membership.head())

In [ ]:
# -------------------------------------------------------------------
# CANDIDATE CLUSTERS
# -------------------------------------------------------------------
def build_candidate_clusters(candidate_union: pd.DataFrame, candidate_raw: pd.DataFrame, cluster_radius_px: float):
    if len(candidate_union) == 0:
        return pd.DataFrame(), candidate_union.copy()

    cluster_rows = []
    union_out = candidate_union.copy()

    for keys, g in union_out.groupby(["dataset_id", "well_id", "crop_id"], sort=True):
        g = g.sort_values("union_id").reset_index(drop=True)
        coords = g[["union_centroid_well_y_px", "union_centroid_well_x_px"]].to_numpy(dtype=np.float32)
        labels = connected_components_radius(coords, cluster_radius_px)

        for comp_id in np.unique(labels):
            idx = np.flatnonzero(labels == comp_id)
            comp = g.iloc[idx].copy().reset_index(drop=True)
            union_ids = comp["union_id"].tolist()
            cluster_id = deterministic_cluster_id(union_ids)
            union_out.loc[g.index[idx], "cluster_id"] = cluster_id

            raw_subset = candidate_raw[candidate_raw["union_id"].isin(union_ids)] if "union_id" in candidate_raw.columns else pd.DataFrame()
            # raw_subset not available yet in candidate_raw; approximate with membership join later
            top2_scores = sorted(comp["top_method_score_norm"].astype(float).tolist(), reverse=True)[:2]
            top2_support = sorted(comp["proposer_support_count"].astype(float).tolist(), reverse=True)[:2]
            score_gap_top2 = float(top2_scores[0] - top2_scores[1]) if len(top2_scores) >= 2 else np.nan
            support_gap_top2 = float(top2_support[0] - top2_support[1]) if len(top2_support) >= 2 else np.nan
            centroid = comp[["union_centroid_well_y_px", "union_centroid_well_x_px"]].to_numpy(dtype=np.float32).mean(axis=0)
            bbox_ymin = float(comp["union_bbox_ymin_px"].min())
            bbox_xmin = float(comp["union_bbox_xmin_px"].min())
            bbox_ymax = float(comp["union_bbox_ymax_px"].max())
            bbox_xmax = float(comp["union_bbox_xmax_px"].max())
            parent_area = float((bbox_ymax - bbox_ymin) * (bbox_xmax - bbox_xmin))
            is_bimodal = bool(len(comp) >= 2 and np.nanmax(comp["proposer_support_count"]) >= 2)

            cluster_rows.append({
                "cluster_id": cluster_id,
                "dataset_id": comp.iloc[0]["dataset_id"],
                "well_id": comp.iloc[0]["well_id"],
                "crop_id": comp.iloc[0]["crop_id"],
                "n_union_candidates": int(len(comp)),
                "n_raw_detections": int(comp["union_n_members"].sum()),
                "cluster_centroid_well_y_px": float(centroid[0]),
                "cluster_centroid_well_x_px": float(centroid[1]),
                "cluster_bbox_ymin_px": bbox_ymin,
                "cluster_bbox_xmin_px": bbox_xmin,
                "cluster_bbox_ymax_px": bbox_ymax,
                "cluster_bbox_xmax_px": bbox_xmax,
                "max_contrast": float(comp["top_method_score_raw"].max()),
                "max_proposer_support": int(comp["proposer_support_count"].max()),
                "score_gap_top2": score_gap_top2,
                "support_gap_top2": support_gap_top2,
                "is_bimodal": is_bimodal,
                "parent_component_area_px": parent_area,
                **{k: comp.iloc[0][k] for k in BASE_PROVENANCE.keys()},
            })

    return pd.DataFrame(cluster_rows), union_out

candidate_clusters, candidate_union = build_candidate_clusters(candidate_union, candidate_raw, CFG["cluster_radius_px"])
log(f"Clusters built: n_clusters={len(candidate_clusters)}")

display(candidate_clusters.head())

In [ ]:
# -------------------------------------------------------------------
# BACK-FILL union_id ON candidate_raw FOR CONVENIENCE
# -------------------------------------------------------------------
if len(candidate_raw) and len(candidate_union_membership):
    candidate_raw = candidate_raw.merge(
        candidate_union_membership[["union_id", "raw_detection_id"]],
        on="raw_detection_id",
        how="left",
    )

display(candidate_raw.head())

In [ ]:
# -------------------------------------------------------------------
# QA SUMMARIES + OVERLAYS
# -------------------------------------------------------------------
detector_counts = (
    candidate_raw.groupby(["method_id", "source_view_id"], dropna=False)
    .size().rename("n_raw").reset_index()
    .sort_values(["method_id", "source_view_id"])
)

union_counts = (
    candidate_union.groupby(["well_id", "crop_id"], dropna=False)
    .size().rename("n_union").reset_index()
    .sort_values(["well_id", "crop_id"])
)

cluster_counts = (
    candidate_clusters.groupby(["well_id", "crop_id"], dropna=False)
    .size().rename("n_clusters").reset_index()
    .sort_values(["well_id", "crop_id"])
)

display(detector_counts.head(20))
display(union_counts.head(20))
display(cluster_counts.head(20))

if CFG["write_overlays"]:
    for _, crop_row in crop_registry.iterrows():
        crop_id = crop_row["crop_id"]
        if crop_id not in CROP_PANEL_CACHE:
            continue
        if "C1_638" in CROP_PANEL_CACHE[crop_id]:
            base = CROP_PANEL_CACHE[crop_id]["C1_638"]["raw"]
        else:
            first_panel = sorted(CROP_PANEL_CACHE[crop_id].keys())[0]
            base = CROP_PANEL_CACHE[crop_id][first_panel]["raw"]

        sub_union = candidate_union[candidate_union["crop_id"] == crop_id].copy()
        if len(sub_union) == 0:
            continue

        fig, ax = plt.subplots(figsize=(7, 7))
        ax.imshow(base, cmap="gray")
        py = sub_union["union_centroid_well_y_px"].to_numpy() - float(crop_row["well_ymin_px"])
        px = sub_union["union_centroid_well_x_px"].to_numpy() - float(crop_row["well_xmin_px"])
        ax.scatter(px, py, s=18, facecolors="none", edgecolors="yellow", linewidths=0.8)
        ax.set_title(f"{crop_id} | union candidates")
        ax.set_axis_off()
        fig.tight_layout()
        out_png = OVERLAY_DIR / f"{crop_id}_union_overlay.png"
        fig.savefig(out_png, dpi=160, bbox_inches="tight")
        plt.close(fig)
        log(f"[overlay] wrote {out_png.name}")

In [ ]:
# -------------------------------------------------------------------
# PERSIST OUTPUTS
# -------------------------------------------------------------------
preprocess_manifest_path = None
candidate_raw_path = None
candidate_union_path = None
candidate_membership_path = None
candidate_clusters_path = None
detector_summary_path = None

if CFG["write_cache_manifest"]:
    preprocess_manifest_path = safe_to_parquet(preprocess_timing_df, TABLES_DIR / f"{RUN_ID}_preprocessing_cache_manifest.parquet")
if CFG["write_candidate_raw"]:
    candidate_raw_path = safe_to_parquet(candidate_raw, TABLES_DIR / f"{RUN_ID}_candidate_raw.parquet")
if CFG["write_candidate_union"]:
    candidate_union_path = safe_to_parquet(candidate_union, TABLES_DIR / f"{RUN_ID}_candidate_union.parquet")
if CFG["write_candidate_membership"]:
    candidate_membership_path = safe_to_parquet(candidate_union_membership, TABLES_DIR / f"{RUN_ID}_candidate_union_membership.parquet")
if CFG["write_candidate_clusters"]:
    candidate_clusters_path = safe_to_parquet(candidate_clusters, TABLES_DIR / f"{RUN_ID}_candidate_clusters.parquet")
if CFG["write_detector_summary"]:
    detector_summary_path = safe_to_parquet(detector_summary_df, TABLES_DIR / f"{RUN_ID}_detector_run_summary.parquet")

run_manifest = {
    "run_id": RUN_ID,
    "notebook_name": "03_candidate_universe_generation.ipynb",
    "repo_root": str(REPO_ROOT),
    "data_root": str(DATA_ROOT),
    "crop_registry_path": str(CROP_REGISTRY_PATH),
    "n_crops": int(len(crop_registry)),
    "n_candidate_raw": int(len(candidate_raw)),
    "n_candidate_union": int(len(candidate_union)),
    "n_candidate_union_membership": int(len(candidate_union_membership)),
    "n_candidate_clusters": int(len(candidate_clusters)),
    "paths": {
        "preprocess_manifest": None if preprocess_manifest_path is None else str(preprocess_manifest_path),
        "candidate_raw": None if candidate_raw_path is None else str(candidate_raw_path),
        "candidate_union": None if candidate_union_path is None else str(candidate_union_path),
        "candidate_union_membership": None if candidate_membership_path is None else str(candidate_membership_path),
        "candidate_clusters": None if candidate_clusters_path is None else str(candidate_clusters_path),
        "detector_summary": None if detector_summary_path is None else str(detector_summary_path),
        "overlay_dir": str(OVERLAY_DIR),
    },
    "config": CFG,
    "base_provenance": BASE_PROVENANCE,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}
run_manifest_path = write_json(run_manifest, MANIFEST_DIR / f"{RUN_ID}_run_manifest.json")
log(f"Manifest written -> {run_manifest_path}")
run_manifest

In [ ]:
# -------------------------------------------------------------------
# EXIT CHECKS
# -------------------------------------------------------------------
required_raw_cols = {
    "raw_detection_id", "run_id", "dataset_id", "well_id", "method_id", "method_family",
    "source_view_id", "score_type", "score_is_calibrated",
    "detection_score_raw", "detection_score_norm",
    "well_y_px", "well_x_px", "coord_frame_primary", "timing_sec",
}
required_union_cols = {
    "union_id", "run_id", "dataset_id", "well_id",
    "union_n_members", "union_centroid_well_y_px", "union_centroid_well_x_px",
    "union_medoid_well_y_px", "union_medoid_well_x_px",
    "union_radius_px", "top_method_id", "top_method_score_raw", "top_method_score_norm",
    "proposer_support_count", "proposer_support_fraction", "family_support_count", "family_set_canonical",
}
required_cluster_cols = {
    "cluster_id", "dataset_id", "well_id", "n_union_candidates",
    "cluster_centroid_well_y_px", "cluster_centroid_well_x_px",
}

missing_raw = sorted(required_raw_cols - set(candidate_raw.columns))
missing_union = sorted(required_union_cols - set(candidate_union.columns))
missing_cluster = sorted(required_cluster_cols - set(candidate_clusters.columns))

assert not missing_raw, f"candidate_raw missing cols: {missing_raw}"
assert not missing_union, f"candidate_union missing cols: {missing_union}"
assert not missing_cluster, f"candidate_clusters missing cols: {missing_cluster}"

assert candidate_raw["raw_detection_id"].is_unique, "raw_detection_id must be unique"
assert candidate_union["union_id"].is_unique, "union_id must be unique"
assert candidate_clusters["cluster_id"].is_unique, "cluster_id must be unique"
assert set(candidate_raw["coord_frame_primary"].dropna().unique()) == {"well"}

log("Notebook 03 exit checks passed.")
print("candidate_raw:", len(candidate_raw))
print("candidate_union:", len(candidate_union))
print("candidate_clusters:", len(candidate_clusters))

In [ ]:
# -------------------------------------------------------------------
# OPTIMIZER DIAGNOSTIC (v9)
# -------------------------------------------------------------------
log("=" * 90)
log("OPTIMIZER DIAGNOSTIC — v9")
log("=" * 90)

_ok = detector_summary_df[detector_summary_df["status"] == "ok"].copy()

if len(_ok) == 0:
    log("[warn] No successful detector runs — nothing to analyse.")
else:
    _stats = _ok.groupby("method_id")["n_points"].agg(
        ["mean", "median", "max", "min", "sum", "count"])
    _stats.columns = ["avg", "median", "max", "min", "total", "n_runs"]
    _stats = _stats.sort_values("total", ascending=False)
    log("\nPer-method summary:")
    display(_stats)

    try:
        _union_stats = candidate_union.groupby(["well_id", "crop_id"]).size()
        log(f"\nUnion candidates per crop: mean={_union_stats.mean():.1f}, "
            f"median={_union_stats.median():.1f}, max={_union_stats.max()}, "
            f"min={_union_stats.min()}")
    except Exception:
        pass

    try:
        _cluster_stats = candidate_clusters.groupby(["well_id", "crop_id"]).size()
        log(f"Clusters per crop:         mean={_cluster_stats.mean():.1f}, "
            f"median={_cluster_stats.median():.1f}, max={_cluster_stats.max()}, "
            f"min={_cluster_stats.min()}")
    except Exception:
        pass


---
# Notebook 04 — Feature Engineering

**Runs in the same kernel as NB03.** All preprocessing cache, union tables,
and provenance are already in memory.

## Feature families (~560 features)

| # | Family | Approx cols | Source |
|---|--------|------------|--------|
| 1 | Patch photometry | ~282 | Image patches at candidate locations |
| 2 | Detector responses | ~78 | Response maps sampled at candidates |
| 3 | Spatial / geometric / cluster | ~40 | KD-tree, cluster table, geometry |
| 4 | Multi-proposer consensus | ~41 | Union membership + raw detections |
| 5 | Barcode / multi-channel | ~40 | Cross-panel consistency |
| 6 | Crop / image regime | ~40 | Global crop statistics |
| 7 | Symmetry / shape | ~16 | Structure tensor, Hu moments, PSF fit |
| 8 | Interaction / derived / rank | ~26 | Cross-feature ratios, ranks |


In [ ]:
# NB04 additional imports (not in NB03)
from scipy.stats import skew, kurtosis
from scipy.spatial import Voronoi

# -------------------------------------------------------------------
# NB04 CONFIG — Feature Engineering
# -------------------------------------------------------------------

NB04_RUN_ID = f"nb04_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}_{hashlib.sha1(f'{RUN_ID}|nb04'.encode()).hexdigest()[:8]}"
log(f"NB04_RUN_ID = {NB04_RUN_ID}")

CFG.update({
    # ── Patch geometry ─────────────────────────────────────────────────────
    "patch_inner_radius_px": 3,
    "patch_ring1_inner_px": 3,
    "patch_ring1_outer_px": 7,
    "patch_ring2_inner_px": 7,
    "patch_ring2_outer_px": 12,
    "min_patch_coverage": 0.5,

    # ── Image maps for photometry ──────────────────────────────────────────
    "photometry_maps": ["raw", "norm01", "white", "peak_bg", "z_local", "highpass"],
    "barcode_panels": ["C1_638", "C4_561", "C4_638", "C5_561", "C5_638"],
    "barcode_panel_maps": ["raw", "norm01"],

    # ── Response maps ──────────────────────────────────────────────────────
    "response_maps": [
        "log_normalized", "matched_filter", "peak_bg",
        "atrous_wavelet_bandpass", "hessian_blobness",
        "radial_symmetry_map", "rolling_ball_residual",
        "opening_recon_residual", "white", "z_local",
    ],

    # ── LoG multi-sigma ────────────────────────────────────────────────────
    "log_sigmas": [2.0, 4.0, 8.0],

    # ── PSF fit ────────────────────────────────────────────────────────────
    "psf_fit_radius_px": 6,

    # ── Spatial features ───────────────────────────────────────────────────
    "neighbor_radii_px": [5.0, 10.0, 20.0],
    "nn_k": 3,

    # ── Hu moments ─────────────────────────────────────────────────────────
    "hu_patch_radius_px": 5,
    "n_hu_moments": 4,

    # ── Structure tensor ───────────────────────────────────────────────────
    "struct_tensor_sigma": 2.0,

    # ── Provenance ─────────────────────────────────────────────────────────
    "feature_registry_version": "v1_nb04",
})

# ── Build crop lookup ──────────────────────────────────────────────────────
CROP_LOOKUP = {}
for _, cr in crop_registry.iterrows():
    CROP_LOOKUP[cr["crop_id"]] = cr

log(f"NB04 config ready. {len(candidate_union)} union candidates across "
    f"{candidate_union['crop_id'].nunique()} crops")
print(f"Patch geometry: inner_r={CFG['patch_inner_radius_px']}, "
      f"ring1=[{CFG['patch_ring1_inner_px']},{CFG['patch_ring1_outer_px']}], "
      f"ring2=[{CFG['patch_ring2_inner_px']},{CFG['patch_ring2_outer_px']}]")


In [ ]:
# -------------------------------------------------------------------
# FEATURE INFRASTRUCTURE
# -------------------------------------------------------------------

def coord_well_to_crop(well_y, well_x, crop_row):
    """Convert well coordinates to crop-local coordinates."""
    cy = well_y - float(crop_row["well_ymin_px"])
    cx = well_x - float(crop_row["well_xmin_px"])
    return cy, cx


def make_circle_mask(radius):
    """Create a boolean circle mask of given radius."""
    d = 2 * radius + 1
    yy, xx = np.ogrid[-radius:radius+1, -radius:radius+1]
    return (yy*yy + xx*xx) <= radius*radius


def make_ring_mask(r_inner, r_outer):
    """Create a boolean annular ring mask."""
    d = 2 * r_outer + 1
    yy, xx = np.ogrid[-r_outer:r_outer+1, -r_outer:r_outer+1]
    dist2 = yy*yy + xx*xx
    return (dist2 > r_inner*r_inner) & (dist2 <= r_outer*r_outer)


# Pre-compute masks (used thousands of times)
_INNER_MASK = make_circle_mask(CFG["patch_inner_radius_px"])
_RING1_MASK = make_ring_mask(CFG["patch_ring1_inner_px"], CFG["patch_ring1_outer_px"])
_RING2_MASK = make_ring_mask(CFG["patch_ring2_inner_px"], CFG["patch_ring2_outer_px"])
_INNER_R = CFG["patch_inner_radius_px"]
_RING1_R = CFG["patch_ring1_outer_px"]
_RING2_R = CFG["patch_ring2_outer_px"]


def extract_masked_patch(image, cy, cx, radius, mask):
    """
    Extract pixel values within a boolean mask centered at (cy, cx).
    
    Returns (values, coverage_fraction) where coverage is the fraction
    of the mask that falls within image bounds.
    """
    H, W = image.shape
    iy, ix = int(round(cy)), int(round(cx))
    
    y0 = iy - radius
    x0 = ix - radius
    y1 = iy + radius + 1
    x1 = ix + radius + 1
    
    # Clamp to image bounds
    py0 = max(0, -y0)
    px0 = max(0, -x0)
    py1 = mask.shape[0] - max(0, y1 - H)
    px1 = mask.shape[1] - max(0, x1 - W)
    
    iy0 = max(0, y0)
    ix0 = max(0, x0)
    iy1 = min(H, y1)
    ix1 = min(W, x1)
    
    if iy1 <= iy0 or ix1 <= ix0:
        return np.array([], dtype=np.float32), 0.0
    
    sub_mask = mask[py0:py1, px0:px1]
    sub_img = image[iy0:iy1, ix0:ix1]
    
    # Ensure shapes match
    min_h = min(sub_mask.shape[0], sub_img.shape[0])
    min_w = min(sub_mask.shape[1], sub_img.shape[1])
    sub_mask = sub_mask[:min_h, :min_w]
    sub_img = sub_img[:min_h, :min_w]
    
    total_mask_pixels = int(mask.sum())
    valid_mask_pixels = int(sub_mask.sum())
    coverage = valid_mask_pixels / max(total_mask_pixels, 1)
    
    values = sub_img[sub_mask].astype(np.float32)
    return values, coverage


def patch_statistics(values):
    """
    Compute summary statistics on a 1D array of patch values.
    Returns dict with: mean, std, max, min, median, skew, kurtosis, sum, count.
    All NaN-safe.
    """
    if len(values) == 0 or not np.any(np.isfinite(values)):
        return {
            "mean": np.nan, "std": np.nan, "max": np.nan, "min": np.nan,
            "median": np.nan, "skew": np.nan, "kurtosis": np.nan,
            "sum": np.nan, "count": 0,
        }
    v = values[np.isfinite(values)]
    if len(v) == 0:
        return {k: np.nan for k in ["mean","std","max","min","median","skew","kurtosis","sum","count"]}
    result = {
        "mean": float(np.mean(v)),
        "std": float(np.std(v)),
        "max": float(np.max(v)),
        "min": float(np.min(v)),
        "median": float(np.median(v)),
        "sum": float(np.sum(v)),
        "count": int(len(v)),
    }
    if len(v) >= 3:
        result["skew"] = float(skew(v, nan_policy="omit"))
        result["kurtosis"] = float(kurtosis(v, nan_policy="omit"))
    else:
        result["skew"] = np.nan
        result["kurtosis"] = np.nan
    return result


def safe_div(a, b, default=np.nan):
    """NaN-safe division."""
    if b is None or (isinstance(b, float) and (np.isnan(b) or abs(b) < 1e-15)):
        return default
    try:
        return float(a) / float(b)
    except (ZeroDivisionError, TypeError, ValueError):
        return default


def bilinear_sample(image, y, x):
    """Sample a 2D image at sub-pixel (y, x) with bilinear interpolation."""
    H, W = image.shape
    if y < 0 or y >= H - 1 or x < 0 or x >= W - 1:
        iy = min(max(int(round(y)), 0), H - 1)
        ix = min(max(int(round(x)), 0), W - 1)
        return float(image[iy, ix])
    y0, x0 = int(y), int(x)
    y1, x1 = min(y0 + 1, H - 1), min(x0 + 1, W - 1)
    dy, dx = y - y0, x - x0
    val = (image[y0, x0] * (1 - dy) * (1 - dx) +
           image[y1, x0] * dy * (1 - dx) +
           image[y0, x1] * (1 - dy) * dx +
           image[y1, x1] * dy * dx)
    return float(val)


def progress_bar(current, total, prefix="", t_start=None, bar_len=40):
    """Print a progress bar to stdout."""
    frac = current / max(total, 1)
    filled = int(bar_len * frac)
    bar = "\u2588" * filled + "\u2591" * (bar_len - filled)
    if t_start is not None:
        elapsed = time.perf_counter() - t_start
        eta = (elapsed / max(current, 1)) * (total - current)
        eta_str = f"{eta:.0f}s" if eta < 3600 else f"{eta/60:.1f}m"
        suffix = f"ETA {eta_str}"
    else:
        suffix = ""
    sys.stdout.write(f"\r{prefix} |{bar}| {current}/{total} ({frac*100:.1f}%) {suffix}  ")
    sys.stdout.flush()


log("Feature infrastructure ready.")
log(f"  Inner mask: {_INNER_MASK.shape}, {int(_INNER_MASK.sum())} pixels")
log(f"  Ring1 mask: {_RING1_MASK.shape}, {int(_RING1_MASK.sum())} pixels")
log(f"  Ring2 mask: {_RING2_MASK.shape}, {int(_RING2_MASK.sum())} pixels")


In [ ]:
# -------------------------------------------------------------------
# FAMILY 1 — PATCH PHOTOMETRY  (~282 features)
#
# For each union candidate, sample image patches at the medoid location
# across multiple image maps and window sizes.
#
# Maps: raw, norm01, white, peak_bg, z_local, highpass (6 aggregate)
# + per-barcode-panel: raw, norm01 (5 panels × 2 maps = 10)
# Windows: inner (r=3), ring1 (3-7), ring2 (7-12)
# Stats: mean, std, max, min, median, skew, kurtosis (inner)
#        mean, std, max, median (rings)
# Derived: contrast, SNR_local, SNR_mad
# -------------------------------------------------------------------

def compute_photometry_for_candidate(cy, cx, panel_cache, crop_row):
    """
    Compute all photometry features for one candidate.
    cy, cx are crop-local coordinates.
    panel_cache is the dict of preprocessing maps for one panel.
    Returns a flat dict of {feature_name: value}.
    """
    feats = {}
    min_cov = CFG["min_patch_coverage"]
    
    # ── Aggregate maps (from the first available barcode panel's cache) ─────
    for map_key in CFG["photometry_maps"]:
        if map_key not in panel_cache:
            continue
        img = panel_cache[map_key]
        
        # Inner patch
        vals, cov = extract_masked_patch(img, cy, cx, _INNER_R, _INNER_MASK)
        if cov >= min_cov and len(vals) > 0:
            stats = patch_statistics(vals)
            for stat_key in ["mean", "std", "max", "min", "median", "skew", "kurtosis"]:
                feats[f"photo_{map_key}_inner_{stat_key}"] = stats[stat_key]
            # Center value (bilinear)
            feats[f"photo_{map_key}_center"] = bilinear_sample(img, cy, cx)
        else:
            for stat_key in ["mean", "std", "max", "min", "median", "skew", "kurtosis"]:
                feats[f"photo_{map_key}_inner_{stat_key}"] = np.nan
            feats[f"photo_{map_key}_center"] = np.nan
        
        # Ring1
        vals_r1, cov_r1 = extract_masked_patch(img, cy, cx, _RING1_R, _RING1_MASK)
        if cov_r1 >= min_cov and len(vals_r1) > 0:
            stats_r1 = patch_statistics(vals_r1)
            for stat_key in ["mean", "std", "max", "median"]:
                feats[f"photo_{map_key}_ring1_{stat_key}"] = stats_r1[stat_key]
        else:
            for stat_key in ["mean", "std", "max", "median"]:
                feats[f"photo_{map_key}_ring1_{stat_key}"] = np.nan
        
        # Ring2
        vals_r2, cov_r2 = extract_masked_patch(img, cy, cx, _RING2_R, _RING2_MASK)
        if cov_r2 >= min_cov and len(vals_r2) > 0:
            stats_r2 = patch_statistics(vals_r2)
            for stat_key in ["mean", "std"]:
                feats[f"photo_{map_key}_ring2_{stat_key}"] = stats_r2[stat_key]
        else:
            for stat_key in ["mean", "std"]:
                feats[f"photo_{map_key}_ring2_{stat_key}"] = np.nan
        
        # Derived: contrast, SNR
        inner_mean = feats.get(f"photo_{map_key}_inner_mean", np.nan)
        ring1_mean = feats.get(f"photo_{map_key}_ring1_mean", np.nan)
        ring1_std = feats.get(f"photo_{map_key}_ring1_std", np.nan)
        feats[f"photo_{map_key}_contrast"] = inner_mean - ring1_mean if np.isfinite(inner_mean) and np.isfinite(ring1_mean) else np.nan
        feats[f"photo_{map_key}_snr_local"] = safe_div(inner_mean - ring1_mean, ring1_std)
    
    # SNR_mad (using local_mad at center)
    if "local_mad" in panel_cache:
        mad_val = bilinear_sample(panel_cache["local_mad"], cy, cx)
        for map_key in CFG["photometry_maps"]:
            contrast = feats.get(f"photo_{map_key}_contrast", np.nan)
            feats[f"photo_{map_key}_snr_mad"] = safe_div(contrast, mad_val)
    
    return feats


def compute_panel_photometry_for_candidate(cy, cx, panel_key, panel_cache):
    """Per-barcode-panel photometry (subset of stats)."""
    feats = {}
    min_cov = CFG["min_patch_coverage"]
    pk = panel_key.replace("_", "").lower()  # e.g. "c1638"
    
    for map_key in CFG["barcode_panel_maps"]:
        if map_key not in panel_cache:
            continue
        img = panel_cache[map_key]
        
        # Center
        feats[f"bpanel_{pk}_{map_key}_center"] = bilinear_sample(img, cy, cx)
        
        # Inner
        vals, cov = extract_masked_patch(img, cy, cx, _INNER_R, _INNER_MASK)
        if cov >= min_cov and len(vals) > 0:
            stats = patch_statistics(vals)
            for sk in ["mean", "std", "max", "min", "median"]:
                feats[f"bpanel_{pk}_{map_key}_inner_{sk}"] = stats[sk]
        else:
            for sk in ["mean", "std", "max", "min", "median"]:
                feats[f"bpanel_{pk}_{map_key}_inner_{sk}"] = np.nan
        
        # Ring1
        vals_r1, cov_r1 = extract_masked_patch(img, cy, cx, _RING1_R, _RING1_MASK)
        if cov_r1 >= min_cov and len(vals_r1) > 0:
            stats_r1 = patch_statistics(vals_r1)
            feats[f"bpanel_{pk}_{map_key}_ring1_mean"] = stats_r1["mean"]
            feats[f"bpanel_{pk}_{map_key}_ring1_std"] = stats_r1["std"]
        else:
            feats[f"bpanel_{pk}_{map_key}_ring1_mean"] = np.nan
            feats[f"bpanel_{pk}_{map_key}_ring1_std"] = np.nan
        
        # Contrast + SNR
        im = feats.get(f"bpanel_{pk}_{map_key}_inner_mean", np.nan)
        rm = feats.get(f"bpanel_{pk}_{map_key}_ring1_mean", np.nan)
        rs = feats.get(f"bpanel_{pk}_{map_key}_ring1_std", np.nan)
        feats[f"bpanel_{pk}_{map_key}_contrast"] = im - rm if np.isfinite(im) and np.isfinite(rm) else np.nan
        feats[f"bpanel_{pk}_{map_key}_snr"] = safe_div(im - rm, rs)
    
    return feats


# ── Execute Family 1 ──────────────────────────────────────────────────────
log("=" * 90)
log("FAMILY 1: Patch photometry")
t0_f1 = time.perf_counter()

F1_ROWS = []
n_total = len(candidate_union)
t_start = time.perf_counter()

for idx, (_, urow) in enumerate(candidate_union.iterrows()):
    uid = urow["union_id"]
    crop_id = urow["crop_id"]
    well_y = float(urow["union_medoid_well_y_px"])
    well_x = float(urow["union_medoid_well_x_px"])
    
    crop_row = CROP_LOOKUP.get(crop_id)
    if crop_row is None or crop_id not in CROP_PANEL_CACHE:
        F1_ROWS.append({"union_id": uid})
        continue
    
    cy, cx = coord_well_to_crop(well_y, well_x, crop_row)
    panels = CROP_PANEL_CACHE[crop_id]
    
    # Use primary barcode panel for aggregate photometry
    primary_panel = None
    for pk in CFG["barcode_panels"]:
        if pk in panels:
            primary_panel = pk
            break
    
    row = {"union_id": uid}
    
    if primary_panel is not None:
        row.update(compute_photometry_for_candidate(cy, cx, panels[primary_panel], crop_row))
    
    # Per-panel photometry
    for pk in CFG["barcode_panels"]:
        if pk in panels:
            row.update(compute_panel_photometry_for_candidate(cy, cx, pk, panels[pk]))
    
    F1_ROWS.append(row)
    
    if (idx + 1) % 50 == 0 or idx == n_total - 1:
        progress_bar(idx + 1, n_total, "F1 Photometry", t_start)

print()
f1_df = pd.DataFrame(F1_ROWS)
t1_f1 = time.perf_counter()
n_feat_cols = len([c for c in f1_df.columns if c != "union_id"])
log(f"Family 1 done: {n_feat_cols} feature columns, {len(f1_df)} rows, {t1_f1 - t0_f1:.1f}s")
log(f"  Non-null rates: min={f1_df.drop(columns='union_id').notna().mean().min():.3f}, "
    f"median={f1_df.drop(columns='union_id').notna().mean().median():.3f}")


In [ ]:
# -------------------------------------------------------------------
# FAMILY 2 — DETECTOR RESPONSE FEATURES (~78 features)
#
# Sample each response map at the candidate location.
# Also: multi-sigma LoG, PSF fit, local prominence.
# -------------------------------------------------------------------

def _gauss2d_nb04(coords, amp, y0, x0, sy, sx, off):
    yy, xx = coords
    return (amp * np.exp(-(((yy-y0)**2)/(2*sy*sy) + ((xx-x0)**2)/(2*sx*sx))) + off).ravel()


def compute_response_features(cy, cx, panel_cache):
    """Detector response features for one candidate."""
    feats = {}
    
    # ── Response map values at center + local stats ────────────────────────
    for map_key in CFG["response_maps"]:
        if map_key not in panel_cache:
            continue
        resp = panel_cache[map_key]
        H, W = resp.shape
        iy, ix = int(round(cy)), int(round(cx))
        
        # Center value
        feats[f"resp_{map_key}_center"] = bilinear_sample(resp, cy, cx)
        
        # Max and mean in 3px radius
        r = 3
        y0m = max(0, iy - r); y1m = min(H, iy + r + 1)
        x0m = max(0, ix - r); x1m = min(W, ix + r + 1)
        patch = resp[y0m:y1m, x0m:x1m]
        if patch.size > 0:
            feats[f"resp_{map_key}_max3"] = float(np.max(patch))
            feats[f"resp_{map_key}_mean3"] = float(np.mean(patch))
            # Local rank: fraction of 7px neighborhood below center value
            r7 = 7
            y07 = max(0, iy - r7); y17 = min(H, iy + r7 + 1)
            x07 = max(0, ix - r7); x17 = min(W, ix + r7 + 1)
            nbhd = resp[y07:y17, x07:x17]
            center_val = feats[f"resp_{map_key}_center"]
            feats[f"resp_{map_key}_local_rank"] = float(np.mean(nbhd <= center_val)) if nbhd.size > 0 else np.nan
        else:
            feats[f"resp_{map_key}_max3"] = np.nan
            feats[f"resp_{map_key}_mean3"] = np.nan
            feats[f"resp_{map_key}_local_rank"] = np.nan
    
    # ── Multi-sigma LoG ───────────────────────────────────────────────────
    norm_img = panel_cache.get("norm01")
    if norm_img is not None:
        for sigma in CFG["log_sigmas"]:
            log_resp = -(sigma ** 2) * ndi.gaussian_laplace(norm_img.astype(np.float32), sigma=sigma)
            feats[f"resp_log_sigma{sigma:.0f}_center"] = bilinear_sample(log_resp, cy, cx)
    
    # ── PSF fit (2D Gaussian on norm01) ───────────────────────────────────
    if norm_img is not None:
        rad = CFG["psf_fit_radius_px"]
        iy, ix = int(round(cy)), int(round(cx))
        H, W = norm_img.shape
        y0 = max(0, iy - rad); y1 = min(H, iy + rad + 1)
        x0 = max(0, ix - rad); x1 = min(W, ix + rad + 1)
        patch = norm_img[y0:y1, x0:x1]
        
        if patch.shape[0] >= 5 and patch.shape[1] >= 5:
            yy, xx = np.indices(patch.shape)
            p0 = [float(patch.max() - patch.min()),
                  float(cy - y0), float(cx - x0), 2.5, 2.5, float(patch.min())]
            bounds = ([0, 0, 0, 0.5, 0.5, -1],
                      [10, patch.shape[0]-1, patch.shape[1]-1, 12.0, 12.0, 2])
            try:
                popt, _ = curve_fit(_gauss2d_nb04, (yy, xx), patch.ravel(),
                                    p0=p0, bounds=bounds, maxfev=3000)
                amp, fy, fx, sy, sx, off = popt
                fitted = _gauss2d_nb04((yy, xx), *popt).reshape(patch.shape)
                residual = np.sqrt(np.mean((patch - fitted) ** 2))
                feats["psf_amplitude"] = float(amp)
                feats["psf_sigma_y"] = float(sy)
                feats["psf_sigma_x"] = float(sx)
                feats["psf_sigma_mean"] = float((sy + sx) / 2)
                feats["psf_aspect_ratio"] = float(max(sy, sx) / max(min(sy, sx), 0.01))
                feats["psf_offset"] = float(off)
                feats["psf_residual_norm"] = float(residual / max(amp, 1e-6))
                feats["psf_fit_success"] = 1.0
            except Exception:
                for k in ["psf_amplitude", "psf_sigma_y", "psf_sigma_x", "psf_sigma_mean",
                          "psf_aspect_ratio", "psf_offset", "psf_residual_norm"]:
                    feats[k] = np.nan
                feats["psf_fit_success"] = 0.0
        else:
            for k in ["psf_amplitude", "psf_sigma_y", "psf_sigma_x", "psf_sigma_mean",
                      "psf_aspect_ratio", "psf_offset", "psf_residual_norm"]:
                feats[k] = np.nan
            feats["psf_fit_success"] = 0.0
    
    return feats


# ── Execute Family 2 ──────────────────────────────────────────────────────
log("=" * 90)
log("FAMILY 2: Detector responses")
t0_f2 = time.perf_counter()

F2_ROWS = []
n_total = len(candidate_union)
t_start = time.perf_counter()

for idx, (_, urow) in enumerate(candidate_union.iterrows()):
    uid = urow["union_id"]
    crop_id = urow["crop_id"]
    well_y = float(urow["union_medoid_well_y_px"])
    well_x = float(urow["union_medoid_well_x_px"])
    
    crop_row = CROP_LOOKUP.get(crop_id)
    if crop_row is None or crop_id not in CROP_PANEL_CACHE:
        F2_ROWS.append({"union_id": uid})
        continue
    
    cy, cx = coord_well_to_crop(well_y, well_x, crop_row)
    panels = CROP_PANEL_CACHE[crop_id]
    
    primary_panel = None
    for pk in CFG["barcode_panels"]:
        if pk in panels:
            primary_panel = pk
            break
    
    row = {"union_id": uid}
    if primary_panel is not None:
        row.update(compute_response_features(cy, cx, panels[primary_panel]))
    F2_ROWS.append(row)
    
    if (idx + 1) % 100 == 0 or idx == n_total - 1:
        progress_bar(idx + 1, n_total, "F2 Responses", t_start)

print()
f2_df = pd.DataFrame(F2_ROWS)
t1_f2 = time.perf_counter()
n_f2 = len([c for c in f2_df.columns if c != "union_id"])
log(f"Family 2 done: {n_f2} feature columns, {t1_f2 - t0_f2:.1f}s")


In [ ]:
# -------------------------------------------------------------------
# FAMILY 3 — SPATIAL / GEOMETRIC / CLUSTER (~40 features)
#
# KD-tree-based neighbor features, border distances, cluster stats.
# No image data needed — purely from union + cluster tables.
# -------------------------------------------------------------------

def compute_spatial_features_for_crop(crop_union_df, crop_clusters_df, crop_row):
    """
    Compute spatial features for ALL union candidates in one crop at once.
    Returns DataFrame with union_id + spatial feature columns.
    """
    if len(crop_union_df) == 0:
        return pd.DataFrame()
    
    coords = crop_union_df[["union_medoid_well_y_px", "union_medoid_well_x_px"]].to_numpy(dtype=np.float64)
    scores = crop_union_df["top_method_score_norm"].to_numpy(dtype=np.float64)
    uids = crop_union_df["union_id"].tolist()
    n = len(coords)
    
    crop_H = float(crop_row["well_ymax_px"] - crop_row["well_ymin_px"])
    crop_W = float(crop_row["well_xmax_px"] - crop_row["well_xmin_px"])
    crop_area = crop_H * crop_W
    crop_ymin = float(crop_row["well_ymin_px"])
    crop_xmin = float(crop_row["well_xmin_px"])
    crop_ymax = float(crop_row["well_ymax_px"])
    crop_xmax = float(crop_row["well_xmax_px"])
    
    # ── KD-tree ────────────────────────────────────────────────────────────
    tree = cKDTree(coords) if n >= 2 else None
    
    rows = []
    for i in range(n):
        f = {"union_id": uids[i]}
        y, x = coords[i]
        
        # Neighbor counts at various radii
        for r in CFG["neighbor_radii_px"]:
            if tree is not None:
                count = len(tree.query_ball_point(coords[i], r)) - 1  # exclude self
            else:
                count = 0
            f[f"n_neighbors_{int(r)}px"] = count
        
        # K nearest neighbor distances
        if tree is not None and n >= CFG["nn_k"] + 1:
            dists, _ = tree.query(coords[i], k=CFG["nn_k"] + 1)  # +1 for self
            for k_idx in range(CFG["nn_k"]):
                f[f"nn{k_idx+1}_distance_px"] = float(dists[k_idx + 1])
        else:
            for k_idx in range(CFG["nn_k"]):
                f[f"nn{k_idx+1}_distance_px"] = np.nan
        
        # Local density
        r_density = 10.0
        n_in_r = f.get("n_neighbors_10px", 0)
        f["local_density_10px"] = n_in_r / (math.pi * r_density**2)
        
        # Nearest stronger neighbor
        stronger_mask = scores > scores[i]
        if np.any(stronger_mask):
            stronger_coords = coords[stronger_mask]
            dists_stronger = np.sqrt(np.sum((stronger_coords - coords[i])**2, axis=1))
            nearest_idx = np.argmin(dists_stronger)
            f["nearest_stronger_dist_px"] = float(dists_stronger[nearest_idx])
            f["nearest_stronger_score_ratio"] = safe_div(scores[i], scores[stronger_mask][nearest_idx])
        else:
            f["nearest_stronger_dist_px"] = np.nan
            f["nearest_stronger_score_ratio"] = np.nan
            f["is_crop_peak"] = 1.0
        
        # NMS margin
        f["nms_margin_px"] = f.get("nearest_stronger_dist_px", np.nan)
        
        # Border distance
        border_y = min(y - crop_ymin, crop_ymax - y)
        border_x = min(x - crop_xmin, crop_xmax - x)
        f["border_distance_px"] = max(min(border_y, border_x), 0.0)
        f["border_distance_frac"] = safe_div(f["border_distance_px"], min(crop_H, crop_W) / 2)
        
        rows.append(f)
    
    result = pd.DataFrame(rows)
    
    # ── Cluster features (join from cluster table) ─────────────────────────
    if len(crop_clusters_df) > 0 and "cluster_id" in crop_union_df.columns:
        cluster_feats = crop_clusters_df.rename(columns={
            "n_union_candidates": "cluster_n_union",
            "n_raw_detections": "cluster_n_raw",
            "parent_component_area_px": "cluster_area_px",
            "max_proposer_support": "cluster_max_support",
        })[["cluster_id", "cluster_n_union", "cluster_n_raw",
            "cluster_area_px", "cluster_max_support"]].copy()
        
        uid_to_cluster = crop_union_df[["union_id", "cluster_id"]].copy()
        result = result.merge(uid_to_cluster, on="union_id", how="left")
        result = result.merge(cluster_feats, on="cluster_id", how="left")
        
        # Rank within cluster
        if "cluster_id" in result.columns:
            result = result.merge(
                crop_union_df[["union_id", "top_method_score_norm"]],
                on="union_id", how="left"
            )
            ranks = []
            for cid, grp in result.groupby("cluster_id"):
                if pd.isna(cid):
                    for uid in grp["union_id"]:
                        ranks.append({"union_id": uid, "cluster_rank_by_score": np.nan,
                                      "is_cluster_peak": np.nan})
                else:
                    sorted_grp = grp.sort_values("top_method_score_norm", ascending=False)
                    for rank, (_, r) in enumerate(sorted_grp.iterrows()):
                        ranks.append({"union_id": r["union_id"],
                                      "cluster_rank_by_score": rank + 1,
                                      "is_cluster_peak": 1.0 if rank == 0 else 0.0})
            rank_df = pd.DataFrame(ranks)
            result = result.drop(columns=["top_method_score_norm", "cluster_id"], errors="ignore")
            result = result.merge(rank_df, on="union_id", how="left")
    
    # Candidate density in crop
    result["crop_candidate_density"] = n / max(crop_area, 1.0)
    result["crop_n_candidates"] = n
    
    return result


# ── Execute Family 3 ──────────────────────────────────────────────────────
log("=" * 90)
log("FAMILY 3: Spatial / geometric / cluster")
t0_f3 = time.perf_counter()

F3_PARTS = []
crops = candidate_union["crop_id"].unique()
t_start = time.perf_counter()

for ci, cid in enumerate(crops):
    crop_u = candidate_union[candidate_union["crop_id"] == cid].copy()
    crop_cl = candidate_clusters[candidate_clusters["crop_id"] == cid].copy() if "crop_id" in candidate_clusters.columns else pd.DataFrame()
    crop_row = CROP_LOOKUP.get(cid)
    if crop_row is None:
        continue
    part = compute_spatial_features_for_crop(crop_u, crop_cl, crop_row)
    if len(part) > 0:
        F3_PARTS.append(part)
    progress_bar(ci + 1, len(crops), "F3 Spatial", t_start)

print()
f3_df = pd.concat(F3_PARTS, ignore_index=True) if F3_PARTS else pd.DataFrame(columns=["union_id"])
t1_f3 = time.perf_counter()
n_f3 = len([c for c in f3_df.columns if c != "union_id"])
log(f"Family 3 done: {n_f3} feature columns, {t1_f3 - t0_f3:.1f}s")


In [ ]:
# -------------------------------------------------------------------
# FAMILY 4 — MULTI-PROPOSER CONSENSUS (~41 features)
#
# Per-detector binary votes, family votes, support stats, consensus spread.
# Derived from candidate_union, candidate_union_membership, candidate_raw.
# -------------------------------------------------------------------

log("=" * 90)
log("FAMILY 4: Multi-proposer consensus")
t0_f4 = time.perf_counter()

# ── Get all unique method_ids and families ─────────────────────────────────
ALL_METHODS = sorted(candidate_raw["method_id"].unique().tolist())
ALL_FAMILIES = sorted(candidate_raw["method_family"].unique().tolist())
log(f"  Methods: {len(ALL_METHODS)}, Families: {len(ALL_FAMILIES)}")

# ── Build membership lookup: union_id → set of method_ids, family_ids ──────
# Also: union_id → per-method max score
membership_with_method = candidate_union_membership.merge(
    candidate_raw[["raw_detection_id", "method_id", "method_family", "detection_score_norm"]],
    on="raw_detection_id", how="left"
)

# Per-union aggregations
union_method_groups = membership_with_method.groupby("union_id")

# Pre-compute per-union method sets and scores
_method_sets = {}
_family_sets = {}
_method_scores = {}
_member_coords_std = {}

for uid, grp in union_method_groups:
    _method_sets[uid] = set(grp["method_id"].dropna().unique())
    _family_sets[uid] = set(grp["method_family"].dropna().unique())
    # Per-family max score
    scores_by_family = {}
    for fam, fgrp in grp.groupby("method_family"):
        scores_by_family[fam] = float(fgrp["detection_score_norm"].max())
    _method_scores[uid] = scores_by_family

# Consensus spread from membership coordinates
_member_spread = {}
if "member_distance_to_centroid_px" in candidate_union_membership.columns:
    for uid, grp in candidate_union_membership.groupby("union_id"):
        dists = grp["member_distance_to_centroid_px"].dropna().values
        _member_spread[uid] = float(np.std(dists)) if len(dists) > 1 else 0.0

# ── Build feature rows ────────────────────────────────────────────────────
F4_ROWS = []
n_total = len(candidate_union)
t_start = time.perf_counter()

for idx, (_, urow) in enumerate(candidate_union.iterrows()):
    uid = urow["union_id"]
    f = {"union_id": uid}
    
    # Already in union table
    f["proposer_support_count"] = int(urow.get("proposer_support_count", 0))
    f["proposer_support_fraction"] = float(urow.get("proposer_support_fraction", 0))
    f["family_support_count"] = int(urow.get("family_support_count", 0))
    f["union_n_members"] = int(urow.get("union_n_members", 0))
    f["union_radius_px"] = float(urow.get("union_radius_px", 0))
    
    # Per-method binary votes
    methods_present = _method_sets.get(uid, set())
    for m in ALL_METHODS:
        f[f"vote_{m}"] = 1.0 if m in methods_present else 0.0
    
    # Per-family binary votes
    families_present = _family_sets.get(uid, set())
    for fam in ALL_FAMILIES:
        f[f"vote_family_{fam}"] = 1.0 if fam in families_present else 0.0
    
    # Per-family max scores
    fam_scores = _method_scores.get(uid, {})
    for fam in ALL_FAMILIES:
        f[f"top_score_{fam}"] = fam_scores.get(fam, 0.0)
    
    # Consensus spread
    f["consensus_centroid_spread_px"] = _member_spread.get(uid, 0.0)
    
    # Score CV among members
    member_grp = membership_with_method[membership_with_method["union_id"] == uid] if uid in _method_sets else pd.DataFrame()
    if len(member_grp) > 1:
        sc = member_grp["detection_score_norm"].dropna().values
        f["consensus_score_cv"] = safe_div(float(np.std(sc)), float(np.mean(sc)))
    else:
        f["consensus_score_cv"] = 0.0
    
    F4_ROWS.append(f)
    
    if (idx + 1) % 200 == 0 or idx == n_total - 1:
        progress_bar(idx + 1, n_total, "F4 Consensus", t_start)

print()
f4_df = pd.DataFrame(F4_ROWS)
t1_f4 = time.perf_counter()
n_f4 = len([c for c in f4_df.columns if c != "union_id"])
log(f"Family 4 done: {n_f4} feature columns, {t1_f4 - t0_f4:.1f}s")


In [ ]:
# -------------------------------------------------------------------
# FAMILY 5 — BARCODE / MULTI-CHANNEL (~40 features)
#
# Cross-panel consistency: how consistent is the spot signal across
# the 5 barcode panels? True spots appear in multiple panels;
# artifacts typically appear in only one.
# -------------------------------------------------------------------

log("=" * 90)
log("FAMILY 5: Barcode / multi-channel")
t0_f5 = time.perf_counter()

F5_ROWS = []
n_total = len(candidate_union)
t_start = time.perf_counter()
bp = CFG["barcode_panels"]

for idx, (_, urow) in enumerate(candidate_union.iterrows()):
    uid = urow["union_id"]
    crop_id = urow["crop_id"]
    well_y = float(urow["union_medoid_well_y_px"])
    well_x = float(urow["union_medoid_well_x_px"])
    
    f = {"union_id": uid}
    crop_row = CROP_LOOKUP.get(crop_id)
    panels = CROP_PANEL_CACHE.get(crop_id, {})
    
    if crop_row is None:
        F5_ROWS.append(f)
        continue
    
    cy, cx = coord_well_to_crop(well_y, well_x, crop_row)
    
    # Collect per-panel center values and contrasts
    panel_centers = {}
    panel_contrasts = {}
    panel_snrs = {}
    
    for pk in bp:
        if pk not in panels:
            continue
        cache = panels[pk]
        norm_img = cache.get("norm01")
        if norm_img is None:
            continue
        
        center_val = bilinear_sample(norm_img, cy, cx)
        panel_centers[pk] = center_val
        
        # Quick inner-ring contrast
        vals_in, cov_in = extract_masked_patch(norm_img, cy, cx, _INNER_R, _INNER_MASK)
        vals_rg, cov_rg = extract_masked_patch(norm_img, cy, cx, _RING1_R, _RING1_MASK)
        if cov_in > 0.5 and cov_rg > 0.5 and len(vals_in) > 0 and len(vals_rg) > 0:
            inner_m = float(np.mean(vals_in))
            ring_m = float(np.mean(vals_rg))
            ring_s = float(np.std(vals_rg))
            panel_contrasts[pk] = inner_m - ring_m
            panel_snrs[pk] = safe_div(inner_m - ring_m, ring_s)
        else:
            panel_contrasts[pk] = np.nan
            panel_snrs[pk] = np.nan
    
    # ── Cross-panel aggregation ────────────────────────────────────────────
    contrasts = np.array([v for v in panel_contrasts.values() if np.isfinite(v)])
    centers = np.array([v for v in panel_centers.values() if np.isfinite(v)])
    snrs_arr = np.array([v for v in panel_snrs.values() if np.isfinite(v)])
    
    f["barcode_n_panels_present"] = len(panel_centers)
    
    if len(contrasts) >= 2:
        f["barcode_mean_contrast"] = float(np.mean(contrasts))
        f["barcode_std_contrast"] = float(np.std(contrasts))
        f["barcode_max_contrast"] = float(np.max(contrasts))
        f["barcode_min_contrast"] = float(np.min(contrasts))
        f["barcode_contrast_cv"] = safe_div(float(np.std(contrasts)), float(np.mean(contrasts)))
    else:
        for k in ["barcode_mean_contrast", "barcode_std_contrast", "barcode_max_contrast",
                   "barcode_min_contrast", "barcode_contrast_cv"]:
            f[k] = np.nan
    
    if len(centers) >= 2:
        f["barcode_consistency_score"] = 1.0 - safe_div(float(np.std(centers)), float(np.mean(centers)), 1.0)
        f["barcode_center_mean"] = float(np.mean(centers))
        f["barcode_center_std"] = float(np.std(centers))
    else:
        f["barcode_consistency_score"] = np.nan
        f["barcode_center_mean"] = np.nan
        f["barcode_center_std"] = np.nan
    
    # N panels above threshold
    if len(snrs_arr) > 0:
        f["barcode_n_panels_snr_gt2"] = int(np.sum(snrs_arr > 2.0))
        f["barcode_n_panels_snr_gt5"] = int(np.sum(snrs_arr > 5.0))
        f["barcode_mean_snr"] = float(np.mean(snrs_arr))
        f["barcode_max_snr"] = float(np.max(snrs_arr))
    else:
        f["barcode_n_panels_snr_gt2"] = 0
        f["barcode_n_panels_snr_gt5"] = 0
        f["barcode_mean_snr"] = np.nan
        f["barcode_max_snr"] = np.nan
    
    # 561 vs 638 ratio
    vals_561 = [panel_contrasts.get(pk, np.nan) for pk in bp if "561" in pk]
    vals_638 = [panel_contrasts.get(pk, np.nan) for pk in bp if "638" in pk]
    mean_561 = np.nanmean(vals_561) if any(np.isfinite(v) for v in vals_561) else np.nan
    mean_638 = np.nanmean(vals_638) if any(np.isfinite(v) for v in vals_638) else np.nan
    f["barcode_561_638_ratio"] = safe_div(mean_561, mean_638)
    
    F5_ROWS.append(f)
    
    if (idx + 1) % 200 == 0 or idx == n_total - 1:
        progress_bar(idx + 1, n_total, "F5 Barcode", t_start)

print()
f5_df = pd.DataFrame(F5_ROWS)
t1_f5 = time.perf_counter()
n_f5 = len([c for c in f5_df.columns if c != "union_id"])
log(f"Family 5 done: {n_f5} feature columns, {t1_f5 - t0_f5:.1f}s")


In [ ]:
# -------------------------------------------------------------------
# FAMILY 6 — CROP / IMAGE REGIME (~40 features)
#
# Global crop-level statistics. Same value for all candidates in a crop.
# Computed once per crop, then broadcast.
# -------------------------------------------------------------------

log("=" * 90)
log("FAMILY 6: Crop / image regime")
t0_f6 = time.perf_counter()

CROP_REGIME_CACHE = {}

for crop_id, panels in CROP_PANEL_CACHE.items():
    f = {"crop_id": crop_id}
    
    # Use primary barcode panel for aggregate stats
    primary = None
    for pk in CFG["barcode_panels"]:
        if pk in panels:
            primary = pk
            break
    
    if primary is not None:
        cache = panels[primary]
        norm_img = cache["norm01"]
        raw_img = cache["raw"]
        local_mad_img = cache.get("local_mad")
        
        f["crop_background_mean"] = float(np.mean(norm_img))
        f["crop_background_std"] = float(np.std(norm_img))
        f["crop_dynamic_range"] = float(np.percentile(norm_img, 99) - np.percentile(norm_img, 1))
        f["crop_raw_dynamic_range"] = float(np.percentile(raw_img, 99) - np.percentile(raw_img, 1))
        
        if local_mad_img is not None:
            f["crop_noise_sigma"] = float(np.median(local_mad_img))
        else:
            f["crop_noise_sigma"] = np.nan
        
        # Entropy
        hist, _ = np.histogram(norm_img.ravel(), bins=256, range=(0, 1))
        hist = hist / max(hist.sum(), 1)
        hist = hist[hist > 0]
        f["crop_entropy"] = float(-np.sum(hist * np.log2(hist)))
        
        # Edge density (Canny)
        try:
            from skimage.feature import canny
            edges = canny(norm_img, sigma=1.5)
            f["crop_edge_density"] = float(edges.sum() / max(edges.size, 1))
        except Exception:
            f["crop_edge_density"] = np.nan
        
        # Illumination nonuniformity
        smooth = ndi.gaussian_filter(norm_img, sigma=30)
        f["crop_illumination_nonuniformity"] = safe_div(float(np.std(smooth)), float(np.mean(smooth)))
    
    # Per-panel noise and dynamic range
    for pk in CFG["barcode_panels"]:
        pk_short = pk.replace("_", "").lower()
        if pk in panels:
            pc = panels[pk]
            f[f"crop_{pk_short}_noise_sigma"] = float(np.median(pc["local_mad"])) if "local_mad" in pc else np.nan
            f[f"crop_{pk_short}_dynamic_range"] = float(
                np.percentile(pc["norm01"], 99) - np.percentile(pc["norm01"], 1))
            f[f"crop_{pk_short}_background_mean"] = float(np.mean(pc["norm01"]))
        else:
            f[f"crop_{pk_short}_noise_sigma"] = np.nan
            f[f"crop_{pk_short}_dynamic_range"] = np.nan
            f[f"crop_{pk_short}_background_mean"] = np.nan
    
    CROP_REGIME_CACHE[crop_id] = f

# Broadcast to all union candidates
F6_ROWS = []
for _, urow in candidate_union.iterrows():
    uid = urow["union_id"]
    crop_id = urow["crop_id"]
    regime = CROP_REGIME_CACHE.get(crop_id, {})
    row = {"union_id": uid}
    row.update({k: v for k, v in regime.items() if k != "crop_id"})
    F6_ROWS.append(row)

f6_df = pd.DataFrame(F6_ROWS)
t1_f6 = time.perf_counter()
n_f6 = len([c for c in f6_df.columns if c != "union_id"])
log(f"Family 6 done: {n_f6} feature columns, {t1_f6 - t0_f6:.1f}s")


In [ ]:
# -------------------------------------------------------------------
# FAMILY 7 — SYMMETRY / SHAPE (~16 features)
#
# Structure tensor eccentricity, Hu moments, radial symmetry.
# PSF fit metrics are already in Family 2.
# -------------------------------------------------------------------

log("=" * 90)
log("FAMILY 7: Symmetry / shape")
t0_f7 = time.perf_counter()

def compute_shape_features(cy, cx, panel_cache):
    """Shape features for one candidate."""
    feats = {}
    norm_img = panel_cache.get("norm01")
    if norm_img is None:
        return feats
    
    H, W = norm_img.shape
    iy, ix = int(round(cy)), int(round(cx))
    
    # ── Radial symmetry score ──────────────────────────────────────────────
    rs_map = panel_cache.get("radial_symmetry_map")
    if rs_map is not None:
        feats["shape_radial_symmetry"] = bilinear_sample(rs_map, cy, cx)
        # Residual: compare center value to ring mean
        vals_in, _ = extract_masked_patch(rs_map, cy, cx, _INNER_R, _INNER_MASK)
        vals_rg, _ = extract_masked_patch(rs_map, cy, cx, _RING1_R, _RING1_MASK)
        if len(vals_in) > 0 and len(vals_rg) > 0:
            feats["shape_radial_symmetry_residual"] = float(np.mean(vals_in)) - float(np.mean(vals_rg))
        else:
            feats["shape_radial_symmetry_residual"] = np.nan
    
    # ── Structure tensor eccentricity ──────────────────────────────────────
    sigma_st = CFG["struct_tensor_sigma"]
    smooth = ndi.gaussian_filter(norm_img.astype(np.float64), sigma=sigma_st)
    gy, gx = np.gradient(smooth)
    Jxx = ndi.gaussian_filter(gx * gx, sigma=sigma_st)
    Jyy = ndi.gaussian_filter(gy * gy, sigma=sigma_st)
    Jxy = ndi.gaussian_filter(gx * gy, sigma=sigma_st)
    
    jxx = float(Jxx[min(iy, H-1), min(ix, W-1)])
    jyy = float(Jyy[min(iy, H-1), min(ix, W-1)])
    jxy = float(Jxy[min(iy, H-1), min(ix, W-1)])
    
    trace = jxx + jyy
    det_val = jxx * jyy - jxy * jxy
    disc = max(trace * trace - 4.0 * det_val, 0.0)
    lam1 = 0.5 * (trace + math.sqrt(disc))
    lam2 = 0.5 * (trace - math.sqrt(disc))
    lam_max = max(abs(lam1), abs(lam2))
    lam_min = min(abs(lam1), abs(lam2))
    
    feats["shape_eccentricity"] = safe_div(lam_min, lam_max) if lam_max > 1e-12 else np.nan
    feats["shape_structure_trace"] = trace
    feats["shape_structure_det"] = det_val
    
    # ── Hu moments on 11×11 patch ──────────────────────────────────────────
    r_hu = CFG["hu_patch_radius_px"]
    y0 = max(0, iy - r_hu); y1 = min(H, iy + r_hu + 1)
    x0 = max(0, ix - r_hu); x1 = min(W, ix + r_hu + 1)
    patch = norm_img[y0:y1, x0:x1]
    
    if patch.shape[0] >= 5 and patch.shape[1] >= 5:
        try:
            # Threshold patch for moment computation
            thr = float(np.mean(patch))
            binary = (patch > thr).astype(np.uint8)
            if binary.sum() > 0:
                m = measure.moments_central(binary.astype(float), order=3)
                nm = measure.moments_normalized(m, order=3)
                hu = measure.moments_hu(nm)
                for k in range(min(CFG["n_hu_moments"], len(hu))):
                    # Log-transform Hu moments (they span many orders of magnitude)
                    val = hu[k]
                    feats[f"shape_hu{k+1}"] = float(-np.sign(val) * np.log10(max(abs(val), 1e-30)))
            else:
                for k in range(CFG["n_hu_moments"]):
                    feats[f"shape_hu{k+1}"] = np.nan
        except Exception:
            for k in range(CFG["n_hu_moments"]):
                feats[f"shape_hu{k+1}"] = np.nan
    else:
        for k in range(CFG["n_hu_moments"]):
            feats[f"shape_hu{k+1}"] = np.nan
    
    # ── Circularity: fraction of above-half-max pixels that form a circle ──
    vals_in, cov = extract_masked_patch(norm_img, cy, cx, _INNER_R, _INNER_MASK)
    if cov > 0.5 and len(vals_in) > 3:
        half_max = (float(np.max(vals_in)) + float(np.min(vals_in))) / 2
        above = float(np.sum(vals_in > half_max))
        feats["shape_circularity"] = safe_div(above, float(len(vals_in)))
    else:
        feats["shape_circularity"] = np.nan
    
    return feats


# ── Execute Family 7 ──────────────────────────────────────────────────────
F7_ROWS = []
n_total = len(candidate_union)
t_start = time.perf_counter()

for idx, (_, urow) in enumerate(candidate_union.iterrows()):
    uid = urow["union_id"]
    crop_id = urow["crop_id"]
    well_y = float(urow["union_medoid_well_y_px"])
    well_x = float(urow["union_medoid_well_x_px"])
    
    crop_row = CROP_LOOKUP.get(crop_id)
    panels = CROP_PANEL_CACHE.get(crop_id, {})
    
    row = {"union_id": uid}
    if crop_row is not None:
        cy, cx = coord_well_to_crop(well_y, well_x, crop_row)
        primary = next((pk for pk in CFG["barcode_panels"] if pk in panels), None)
        if primary is not None:
            row.update(compute_shape_features(cy, cx, panels[primary]))
    
    F7_ROWS.append(row)
    
    if (idx + 1) % 100 == 0 or idx == n_total - 1:
        progress_bar(idx + 1, n_total, "F7 Shape", t_start)

print()
f7_df = pd.DataFrame(F7_ROWS)
t1_f7 = time.perf_counter()
n_f7 = len([c for c in f7_df.columns if c != "union_id"])
log(f"Family 7 done: {n_f7} feature columns, {t1_f7 - t0_f7:.1f}s")


In [ ]:
# -------------------------------------------------------------------
# FAMILY 8 — INTERACTION / DERIVED / RANK (~26 features)
#
# PASS 2: These features require the full feature table from families 1-7.
# They compute cross-feature ratios, within-crop percentile ranks,
# and MAD-standardized copies of key features.
# -------------------------------------------------------------------

log("=" * 90)
log("FAMILY 8: Interaction / derived / rank (Pass 2)")
t0_f8 = time.perf_counter()

# ── First, assemble Pass 1 features ───────────────────────────────────────
log("Assembling Pass 1 features for rank computation ...")
pass1_parts = [f1_df, f2_df, f3_df, f4_df, f5_df, f6_df, f7_df]
pass1 = pass1_parts[0]
for part in pass1_parts[1:]:
    if len(part) > 0 and "union_id" in part.columns:
        pass1 = pass1.merge(part, on="union_id", how="left", suffixes=("", "_dup"))
        # Drop duplicates from merge
        dup_cols = [c for c in pass1.columns if c.endswith("_dup")]
        pass1 = pass1.drop(columns=dup_cols)

# Add crop_id for grouping
pass1 = pass1.merge(
    candidate_union[["union_id", "crop_id"]], on="union_id", how="left"
)

log(f"Pass 1 assembled: {len(pass1)} rows, {len(pass1.columns)} columns")

# ── Key features to rank / standardize ─────────────────────────────────────
RANK_FEATURES = [
    ("photo_norm01_contrast", "contrast"),
    ("photo_norm01_snr_local", "snr"),
    ("proposer_support_count", "support"),
    ("resp_log_normalized_center", "log_resp"),
    ("resp_matched_filter_center", "mf_resp"),
    ("barcode_mean_contrast", "barcode_contrast"),
]

F8_ROWS = []

for crop_id, grp in pass1.groupby("crop_id"):
    n_in_crop = len(grp)
    
    for _, row in grp.iterrows():
        f = {"union_id": row["union_id"]}
        
        # ── Percentile ranks within crop ───────────────────────────────────
        for feat_col, short_name in RANK_FEATURES:
            if feat_col in grp.columns:
                vals = grp[feat_col].dropna()
                val = row.get(feat_col, np.nan)
                if np.isfinite(val) and len(vals) > 1:
                    f[f"rank_{short_name}_in_crop"] = float((vals < val).sum()) / len(vals)
                else:
                    f[f"rank_{short_name}_in_crop"] = np.nan
            else:
                f[f"rank_{short_name}_in_crop"] = np.nan
        
        # ── MAD-standardized copies ────────────────────────────────────────
        for feat_col, short_name in RANK_FEATURES[:3]:  # contrast, snr, support
            if feat_col in grp.columns:
                vals = grp[feat_col].dropna()
                val = row.get(feat_col, np.nan)
                if np.isfinite(val) and len(vals) > 2:
                    med = float(np.median(vals))
                    mad = float(np.median(np.abs(vals - med))) / 0.6745
                    f[f"mad_std_{short_name}"] = safe_div(val - med, mad)
                else:
                    f[f"mad_std_{short_name}"] = np.nan
            else:
                f[f"mad_std_{short_name}"] = np.nan
        
        # ── Cross-view ratios ──────────────────────────────────────────────
        raw_contrast = row.get("photo_raw_contrast", np.nan)
        f["ratio_white_over_raw"] = safe_div(row.get("photo_white_contrast", np.nan), raw_contrast)
        f["ratio_peakbg_over_raw"] = safe_div(row.get("photo_peak_bg_contrast", np.nan), raw_contrast)
        f["ratio_zlocal_over_raw"] = safe_div(row.get("photo_z_local_contrast", np.nan), raw_contrast)
        
        # ── Detector response ratios ───────────────────────────────────────
        log_resp = row.get("resp_log_normalized_center", np.nan)
        contrast = row.get("photo_norm01_contrast", np.nan)
        f["ratio_log_over_contrast"] = safe_div(log_resp, contrast)
        
        hess_resp = row.get("resp_hessian_blobness_center", np.nan)
        f["ratio_hessian_over_log"] = safe_div(hess_resp, log_resp)
        
        mf_resp = row.get("resp_matched_filter_center", np.nan)
        f["ratio_mf_over_log"] = safe_div(mf_resp, log_resp)
        
        # ── Support / score interaction ────────────────────────────────────
        support = row.get("proposer_support_count", 0)
        f["support_times_contrast"] = support * contrast if np.isfinite(contrast) else np.nan
        f["support_times_snr"] = support * row.get("photo_norm01_snr_local", np.nan) if np.isfinite(row.get("photo_norm01_snr_local", np.nan)) else np.nan
        
        F8_ROWS.append(f)

f8_df = pd.DataFrame(F8_ROWS)
t1_f8 = time.perf_counter()
n_f8 = len([c for c in f8_df.columns if c != "union_id"])
log(f"Family 8 done: {n_f8} feature columns, {t1_f8 - t0_f8:.1f}s")


In [ ]:
# -------------------------------------------------------------------
# ASSEMBLY + QA
# -------------------------------------------------------------------

log("=" * 90)
log("ASSEMBLING mega_feature_table")
t0_asm = time.perf_counter()

# ── Merge all families ─────────────────────────────────────────────────────
all_families = [f1_df, f2_df, f3_df, f4_df, f5_df, f6_df, f7_df, f8_df]
mega = all_families[0]
for i, part in enumerate(all_families[1:], 2):
    if len(part) > 0 and "union_id" in part.columns:
        mega = mega.merge(part, on="union_id", how="left", suffixes=("", f"_fam{i}_dup"))
        dup_cols = [c for c in mega.columns if "_dup" in c]
        mega = mega.drop(columns=dup_cols)

log(f"Merged {len(all_families)} families: {len(mega)} rows, {len(mega.columns)} columns")

# ── Add coordinates and provenance from union table ────────────────────────
coord_cols = ["union_id", "dataset_id", "well_id", "crop_id", "cluster_id",
              "union_medoid_well_y_px", "union_medoid_well_x_px",
              "union_centroid_well_y_px", "union_centroid_well_x_px"]
coord_df = candidate_union[[c for c in coord_cols if c in candidate_union.columns]].copy()
coord_df = coord_df.rename(columns={
    "union_medoid_well_y_px": "well_y_px",
    "union_medoid_well_x_px": "well_x_px",
})

mega = coord_df.merge(mega, on="union_id", how="left")

# ── in_annotated_crop flag ─────────────────────────────────────────────────
mega["in_annotated_crop"] = mega["crop_id"].notna()
mega["annotation_coverage_status"] = mega["crop_id"].apply(
    lambda x: "in_crop" if pd.notna(x) else "outside_crop"
)

# ── Provenance ─────────────────────────────────────────────────────────────
mega["run_id"] = RUN_ID
mega["nb04_run_id"] = NB04_RUN_ID
mega["project_config_version"] = CFG["project_config_version"]
mega["feature_registry_version"] = CFG["feature_registry_version"]
mega["created_at_utc"] = datetime.now(timezone.utc).isoformat()

# ── Identify feature columns (everything that's not metadata) ──────────────
META_COLS = {"union_id", "dataset_id", "well_id", "crop_id", "cluster_id",
             "well_y_px", "well_x_px", "union_centroid_well_y_px", "union_centroid_well_x_px",
             "in_annotated_crop", "annotation_coverage_status",
             "run_id", "nb04_run_id", "project_config_version",
             "feature_registry_version", "created_at_utc"}
FEATURE_COLS = sorted([c for c in mega.columns if c not in META_COLS])

log(f"\nTotal feature columns: {len(FEATURE_COLS)}")
log(f"Total columns (incl. metadata): {len(mega.columns)}")
log(f"Rows: {len(mega)}")

# ── QA: non-null rates ─────────────────────────────────────────────────────
nn_rates = mega[FEATURE_COLS].notna().mean()
log(f"\nNon-null rates across {len(FEATURE_COLS)} features:")
log(f"  min:    {nn_rates.min():.4f}")
log(f"  median: {nn_rates.median():.4f}")
log(f"  mean:   {nn_rates.mean():.4f}")
log(f"  max:    {nn_rates.max():.4f}")

# Constant columns
constant_cols = [c for c in FEATURE_COLS if mega[c].dropna().nunique() <= 1]
log(f"  Constant columns: {len(constant_cols)}")
if constant_cols:
    log(f"    {constant_cols[:10]}{'...' if len(constant_cols) > 10 else ''}")

# All-NaN columns
all_nan_cols = [c for c in FEATURE_COLS if mega[c].isna().all()]
log(f"  All-NaN columns: {len(all_nan_cols)}")
if all_nan_cols:
    log(f"    {all_nan_cols[:10]}{'...' if len(all_nan_cols) > 10 else ''}")

# dtype summary
dtypes = mega[FEATURE_COLS].dtypes.value_counts()
log(f"\n  Dtype distribution: {dict(dtypes)}")

t1_asm = time.perf_counter()
log(f"\nAssembly done in {t1_asm - t0_asm:.1f}s")


In [ ]:
# -------------------------------------------------------------------
# FEATURE STATS TABLE (§3.12)
# -------------------------------------------------------------------

log("=" * 90)
log("Computing feature_stats.parquet")

feature_stats_rows = []
for col in FEATURE_COLS:
    vals = mega[col].dropna()
    
    # Determine feature group from naming convention
    if col.startswith("photo_"):
        group = "patch_photometry"
    elif col.startswith("bpanel_"):
        group = "patch_photometry"
    elif col.startswith("resp_") or col.startswith("psf_"):
        group = "detector_response"
    elif col.startswith("n_neighbors") or col.startswith("nn") or col.startswith("local_density") or col.startswith("border_") or col.startswith("cluster_") or col.startswith("nearest_") or col.startswith("nms_") or col.startswith("crop_n_") or col.startswith("crop_candidate"):
        group = "spatial_geometric"
    elif col.startswith("vote_") or col.startswith("proposer_") or col.startswith("family_") or col.startswith("union_") or col.startswith("consensus_") or col.startswith("top_score_"):
        group = "consensus"
    elif col.startswith("barcode_"):
        group = "barcode_multichannel"
    elif col.startswith("crop_"):
        group = "crop_regime"
    elif col.startswith("shape_"):
        group = "symmetry_shape"
    elif col.startswith("rank_") or col.startswith("mad_std_") or col.startswith("ratio_") or col.startswith("support_times"):
        group = "interaction_derived"
    else:
        group = "other"
    
    is_core = group in {"patch_photometry", "detector_response", "consensus",
                        "spatial_geometric", "barcode_multichannel"}
    
    feature_stats_rows.append({
        "feature_name": col,
        "feature_group": group,
        "is_core_frozen": is_core,
        "non_null_fraction": float(len(vals)) / max(len(mega), 1),
        "constant_flag": len(vals.unique()) <= 1 if len(vals) > 0 else True,
        "univariate_score": np.nan,  # computed in NB06 with labels
        "correlation_pruned_flag": False,  # computed in NB06
        "importance_metric": np.nan,  # computed in NB06
        "retained_for_model": True,  # default; pruned in NB06
        "feature_registry_version": CFG["feature_registry_version"],
    })

feature_stats = pd.DataFrame(feature_stats_rows)

log(f"\nFeature stats: {len(feature_stats)} features")
log(f"\nPer-group summary:")
display(
    feature_stats.groupby("feature_group").agg(
        n_features=("feature_name", "count"),
        n_core=("is_core_frozen", "sum"),
        avg_non_null=("non_null_fraction", "mean"),
        n_constant=("constant_flag", "sum"),
    ).sort_values("n_features", ascending=False)
)


In [ ]:
# -------------------------------------------------------------------
# PERSIST OUTPUTS
# -------------------------------------------------------------------

log("=" * 90)
log("Persisting outputs")

TABLES_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

mega_path = safe_to_parquet(mega, TABLES_DIR / f"{NB04_RUN_ID}_mega_feature_table.parquet")
stats_path = safe_to_parquet(feature_stats, TABLES_DIR / f"{NB04_RUN_ID}_feature_stats.parquet")

log(f"  mega_feature_table: {mega_path.name} ({len(mega)} rows, {len(mega.columns)} cols)")
log(f"  feature_stats: {stats_path.name} ({len(feature_stats)} rows)")

# ── Manifest ───────────────────────────────────────────────────────────────
manifest = {
    "nb04_run_id": NB04_RUN_ID,
    "nb03_run_id": RUN_ID,
    "n_union_candidates": int(len(mega)),
    "n_feature_columns": int(len(FEATURE_COLS)),
    "n_core_frozen": int(feature_stats["is_core_frozen"].sum()),
    "n_constant_cols": int(feature_stats["constant_flag"].sum()),
    "n_all_nan_cols": int(len(all_nan_cols)),
    "median_non_null_rate": float(feature_stats["non_null_fraction"].median()),
    "paths": {
        "mega_feature_table": str(mega_path),
        "feature_stats": str(stats_path),
    },
    "config": CFG,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}
manifest_path = write_json(manifest, MANIFEST_DIR / f"{NB04_RUN_ID}_run_manifest.json")
log(f"  manifest: {manifest_path.name}")


In [ ]:
# -------------------------------------------------------------------
# EXIT CHECKS
# -------------------------------------------------------------------

log("=" * 90)
log("EXIT CHECKS")

# Schema validation (§3.8)
required_cols = {
    "union_id", "dataset_id", "well_id", "well_y_px", "well_x_px",
    "in_annotated_crop", "annotation_coverage_status",
    "run_id", "project_config_version", "feature_registry_version",
}
missing = sorted(required_cols - set(mega.columns))
assert not missing, f"mega_feature_table missing required columns: {missing}"

# Unique union_id
assert mega["union_id"].is_unique, "union_id must be unique in mega_feature_table"

# No all-NaN feature columns (warning, not assertion — some may legitimately be NaN)
if all_nan_cols:
    log(f"[warn] {len(all_nan_cols)} all-NaN feature columns — these will be pruned in NB06")

# Feature count sanity
assert len(FEATURE_COLS) >= 50, f"Expected >=50 features, got {len(FEATURE_COLS)}"
log(f"  Feature columns: {len(FEATURE_COLS)} (target ~560)")

# Row count matches union table
assert len(mega) == len(candidate_union), \
    f"Row count mismatch: mega={len(mega)} vs union={len(candidate_union)}"

log(f"\n{'=' * 90}")
log(f"NOTEBOOK 04 COMPLETE")
log(f"  mega_feature_table: {len(mega)} rows × {len(mega.columns)} cols")
log(f"  feature columns:    {len(FEATURE_COLS)}")
log(f"  core_frozen:        {int(feature_stats['is_core_frozen'].sum())}")
log(f"  non-null median:    {feature_stats['non_null_fraction'].median():.4f}")
log(f"{'=' * 90}")
